# 09 WNUT-17 External Replication

## Purpose

This notebook provides the final external sensitivity analysis for the dissertation. It evaluates whether the principal architectural and conformal-prediction findings obtained on CoNLL-2003 remain qualitatively observable when the same methodology is applied independently to WNUT-17.

This is **not** a zero-shot CoNLL-2003 → WNUT-17 transfer experiment. Both architectures are trained and calibrated using WNUT-17 data and retain the native WNUT entity ontology:

- `corporation`
- `creative-work`
- `group`
- `location`
- `person`
- `product`

The experiment reproduces the main dissertation methodology using:

1. BERT BIO sequence labelling;
2. the BERT-based SpanNER-style span classifier with maximum span width 4;
3. architecture-specific start- and end-boundary signals;
4. temperature scaling;
5. validation-only development of conformal methods M1–M4;
6. an independent conformal-calibration partition; and
7. final evaluation on the official WNUT-17 test population.

The WNUT study is intentionally narrower than the primary CoNLL-2003 analysis. MC Dropout, binary boundary calibration, controlled character corruption and additional method-development variants are not repeated.

### Final-test provenance

The official WNUT-17 test set was opened only after the original WNUT method specification had been frozen. This submission-ready notebook reproduces that completed evaluation using the already-established methodology; it does not use test results for model selection, calibration or hyperparameter tuning.

### Post-hoc mechanism analysis

A short final section incorporates selected material from the former `09e` notebook. It investigates whether redefining M4's boundary signal around genuine near-tie ambiguity changes its behaviour.

This analysis is explicitly **post-hoc and exploratory**. It reuses the frozen M4 regularisation strengths and the already-evaluated WNUT test candidate universe. Its results are therefore used only to understand the behaviour of M4 and are not treated as independent held-out evidence or as replacements for the primary M1–M4 results.

## 1. Environment and Configuration

The WNUT experiment retains the principal architectural settings used in the primary study:

- random seed: 42;
- encoder: `bert-base-cased`;
- maximum tokenized sequence length: 256;
- SpanNER maximum span width: 4 original words.

The WNUT validation population is subsequently divided into development, probability-calibration and conformal-calibration roles. All methodological decisions are completed before the final test evaluation.

A GPU runtime is recommended for training both architectures.

In [6]:
import gc
import hashlib
import json
import os
import random
from collections import defaultdict
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_CHECKPOINT = "bert-base-cased"
MAX_LENGTH = 256
MAX_SPAN_WIDTH = 4

# The final test has already been opened under the frozen WNUT protocol.
# This flag reproduces that completed evaluation; it must not be used
# to make or revise development decisions.
RUN_FINAL_WNUT_TEST = True

# Selected 09e material is retained only as a post-hoc mechanism diagnostic.
RUN_POSTHOC_TIE_DIAGNOSTIC = True

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = Path(
    "/kaggle/working/09_WNUT17_External_Replication"
)

SPLIT_DIR = OUTPUT_DIR / "split_registry"
EDA_DIR = OUTPUT_DIR / "dataset_summary"
BERT_DIR = OUTPUT_DIR / "bert"
BERT_MODEL_DIR = BERT_DIR / "final_model"
SPANNER_DIR = OUTPUT_DIR / "spanner"
SPANNER_MODEL_DIR = SPANNER_DIR / "final_model"
CALIBRATION_DIR = OUTPUT_DIR / "calibration"
CONFORMAL_DIR = OUTPUT_DIR / "conformal"
TEST_DIR = OUTPUT_DIR / "final_test"
POSTHOC_DIR = OUTPUT_DIR / "posthoc_m4_tie"
SUMMARY_DIR = OUTPUT_DIR / "summary"

for directory in [
    OUTPUT_DIR,
    SPLIT_DIR,
    EDA_DIR,
    BERT_DIR,
    BERT_MODEL_DIR,
    SPANNER_DIR,
    SPANNER_MODEL_DIR,
    CALIBRATION_DIR,
    CONFORMAL_DIR,
    TEST_DIR,
    POSTHOC_DIR,
    SUMMARY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "None",
)
print("Seed:", SEED)
print("Encoder:", MODEL_CHECKPOINT)
print("Maximum sequence length:", MAX_LENGTH)
print("SpanNER maximum width:", MAX_SPAN_WIDTH)
print("Final WNUT test reproduction enabled:", RUN_FINAL_WNUT_TEST)
print("Post-hoc tie diagnostic enabled:", RUN_POSTHOC_TIE_DIAGNOSTIC)

Device: cuda
GPU: Tesla T4
Seed: 42
Encoder: bert-base-cased
Maximum sequence length: 256
SpanNER maximum width: 4
Final WNUT test reproduction enabled: True
Post-hoc tie diagnostic enabled: True


## 2. WNUT-17 Data and Experimental Split

WNUT-17 is used as an external replication dataset while retaining its native six entity types. Both BERT and SpanNER are trained directly on WNUT-17; this is therefore an independent-domain replication rather than zero-shot transfer from CoNLL-2003.

The official training set is used for model fitting. Following the original WNUT experiment, the 1,009 validation sentences are deterministically divided into:

- **505 development sentences** for model selection and M1–M4 development;
- **303 probability-calibration sentences** for temperature scaling;
- **201 conformal-calibration sentences** for conformal threshold estimation.

Exact duplicate token sequences are kept within the same partition to prevent leakage between these roles. The official test set is reserved for the final evaluation.

In [7]:
from datasets import load_dataset

# Load WNUT-17. The fallback mirrors contain the same standard dataset
# and are retained only to make the Kaggle notebook robust to availability.
dataset_sources = [
    "flaitenberger/wnut_17",
    "petaniindo/wnut_17",
    "leondz/wnut_17",
]

wnut = None

for source in dataset_sources:
    try:
        wnut = load_dataset(source)
        break
    except Exception:
        continue

if wnut is None:
    raise RuntimeError("WNUT-17 could not be loaded.")

train_data = wnut["train"]
official_validation = wnut["validation"]

assert len(train_data) == 3394
assert len(official_validation) == 1009


def sentence_hash(tokens):
    """Return a stable identity for an exact token sequence."""
    text = "\u241f".join(tokens)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# Group identical sentences so duplicates cannot cross experimental roles.
groups = defaultdict(list)

for index, example in enumerate(official_validation):
    groups[sentence_hash(example["tokens"])].append(index)

grouped_indices = list(groups.values())

rng = random.Random(SEED)
rng.shuffle(grouped_indices)

target_sizes = {
    "development": 505,
    "probability_calibration": 303,
    "conformal_calibration": 201,
}

split_indices = {name: [] for name in target_sizes}

for group in grouped_indices:
    # Assign the complete duplicate group to the partition with the
    # greatest remaining proportional capacity.
    available = [
        name for name in target_sizes
        if len(split_indices[name]) + len(group) <= target_sizes[name]
    ]

    if available:
        destination = max(
            available,
            key=lambda name:
                (target_sizes[name] - len(split_indices[name]))
                / target_sizes[name],
        )
    else:
        destination = min(
            target_sizes,
            key=lambda name:
                abs(
                    target_sizes[name]
                    - len(split_indices[name])
                    - len(group)
                ),
        )

    split_indices[destination].extend(group)

for name in split_indices:
    split_indices[name].sort()

development_data = official_validation.select(
    split_indices["development"]
)

probability_calibration_data = official_validation.select(
    split_indices["probability_calibration"]
)

conformal_calibration_data = official_validation.select(
    split_indices["conformal_calibration"]
)

# Preserve the validated WNUT experimental populations.
assert len(development_data) == 505
assert len(probability_calibration_data) == 303
assert len(conformal_calibration_data) == 201

split_summary = pd.DataFrame({
    "Population": [
        "Train",
        "Development",
        "Probability calibration",
        "Conformal calibration",
    ],
    "Sentences": [
        len(train_data),
        len(development_data),
        len(probability_calibration_data),
        len(conformal_calibration_data),
    ],
})

display(split_summary)

Repo card metadata block was not found. Setting CardData to empty.


,Population,Sentences
0,Train,3394
1,Development,505
2,Probability calibration,303
3,Conformal calibration,201


## 3. Gold Spans and Evaluation Support

Gold BIO labels are converted to complete typed entity spans using the same conversion rule for both architectures. An incompatible `I-X` tag is treated as the beginning of a new entity of type `X`.

SpanNER considers contiguous spans of at most four original words. Gold entities wider than four words are retained in the native evaluation but are outside the SpanNER candidate support. A matched-width comparison later restricts both architectures to gold entities of width ≤4.


In [8]:
# Recover the native WNUT BIO label mapping directly from the dataset.
label_names = list(train_data.features["ner_tags"].feature.names)

id2label = {
    label_id: label
    for label_id, label in enumerate(label_names)
}

label2id = {
    label: label_id
    for label_id, label in id2label.items()
}

entity_types = [
    label[2:]
    for label in label_names
    if label.startswith("B-")
]


def bio_to_spans(tokens, tags):
    """Convert BIO labels into complete typed entity spans."""
    spans = []
    current = None

    def close_entity(entity, end):
        entity["end"] = end
        entity["text"] = " ".join(
            tokens[entity["start"]:end + 1]
        )
        spans.append(entity)

    for index, tag in enumerate(tags):

        if tag == "O":
            if current is not None:
                close_entity(current, index - 1)
                current = None

        elif tag.startswith("B-"):
            if current is not None:
                close_entity(current, index - 1)

            current = {
                "type": tag[2:],
                "start": index,
            }

        elif tag.startswith("I-"):
            entity_type = tag[2:]

            # Continue only when the I-tag matches the open entity.
            # Otherwise apply the shared BIO repair rule and start
            # a new entity of the indicated type.
            if (
                current is not None
                and current["type"] == entity_type
            ):
                continue

            if current is not None:
                close_entity(current, index - 1)

            current = {
                "type": entity_type,
                "start": index,
            }

    if current is not None:
        close_entity(current, len(tokens) - 1)

    return spans


experiment_splits = {
    "Train": train_data,
    "Development": development_data,
    "Probability calibration": probability_calibration_data,
    "Conformal calibration": conformal_calibration_data,
}

# Store sentence-level gold spans once so both architectures use
# exactly the same evaluation population and boundary definitions.
gold_spans = {}
summary_rows = []

for split_name, split_data in experiment_splits.items():
    split_spans = []

    for example in split_data:
        tags = [
            id2label[int(tag_id)]
            for tag_id in example["ner_tags"]
        ]

        sentence_spans = bio_to_spans(
            example["tokens"],
            tags,
        )

        split_spans.append(sentence_spans)

    gold_spans[split_name] = split_spans

    all_spans = [
        span
        for sentence_spans in split_spans
        for span in sentence_spans
    ]

    width4_count = sum(
        (span["end"] - span["start"] + 1) <= MAX_SPAN_WIDTH
        for span in all_spans
    )

    summary_rows.append({
        "Population": split_name,
        "Gold entities": len(all_spans),
        "Width ≤4": width4_count,
        "Width ≤4 (%)": 100 * width4_count / len(all_spans),
    })

gold_summary = pd.DataFrame(summary_rows)

display(
    gold_summary.round({
        "Width ≤4 (%)": 2,
    })
)

print("\nEntity types:", entity_types)

,Population,Gold entities,Width ≤4,Width ≤4 (%)
0,Train,1975,1934,97.92
1,Development,422,418,99.05
2,Probability calibration,246,244,99.19
3,Conformal calibration,168,164,97.62



Entity types: ['corporation', 'creative-work', 'group', 'location', 'person', 'product']


## 4. BERT BIO Baseline

The WNUT BERT baseline fine-tunes `bert-base-cased` for token-level BIO classification using the native 13-label WNUT ontology. Only the first subword of each original word contributes to the training loss.

Training uses a learning rate of 2×10⁻⁵, batch size 16, weight decay 0.01 and three epochs. The checkpoint with the highest strict typed-span F1 on the development partition is retained.

Performance is reported both on the native WNUT entity population and under a width≤4 restriction matching SpanNER's representable support.


In [9]:
import inspect

bert_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    use_fast=True,
)

bert_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)


def tokenize_and_align_labels(batch):
    """Tokenise pre-split words and supervise only the first subword."""
    encoded = bert_tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []

    for batch_index, word_labels in enumerate(batch["ner_tags"]):
        word_ids = encoded.word_ids(batch_index=batch_index)
        labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                labels.append(-100)
            elif word_id != previous_word_id:
                labels.append(int(word_labels[word_id]))
            else:
                labels.append(-100)

            previous_word_id = word_id

        aligned_labels.append(labels)

    encoded["labels"] = aligned_labels
    return encoded


bert_train_tokenized = train_data.map(
    tokenize_and_align_labels,
    batched=True,
    desc="Tokenising WNUT train",
)

bert_development_tokenized = development_data.map(
    tokenize_and_align_labels,
    batched=True,
    desc="Tokenising WNUT development",
)

bert_data_collator = DataCollatorForTokenClassification(
    tokenizer=bert_tokenizer
)


def span_set_from_tags(tags):
    """Represent one BIO sequence as exact typed-span tuples."""
    spans = bio_to_spans(
        [str(i) for i in range(len(tags))],
        tags,
    )

    return {
        (
            int(span["start"]),
            int(span["end"]),
            str(span["type"]),
        )
        for span in spans
    }


def strict_span_metrics(predicted_sequences, gold_sequences):
    """Calculate micro-averaged exact typed-span metrics."""
    tp = fp = fn = 0

    for predicted_tags, gold_tags in zip(
        predicted_sequences,
        gold_sequences,
    ):
        predicted = span_set_from_tags(predicted_tags)
        gold = span_set_from_tags(gold_tags)

        tp += len(predicted & gold)
        fp += len(predicted - gold)
        fn += len(gold - predicted)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def extract_bert_tag_sequences(logits, labels):
    """Recover one predicted and gold BIO sequence per sentence."""
    if isinstance(logits, tuple):
        logits = logits[0]

    predicted_ids = np.argmax(logits, axis=-1)

    predicted_sequences = []
    gold_sequences = []

    for predicted_row, gold_row in zip(predicted_ids, labels):
        predicted_tags = []
        gold_tags = []

        for predicted_id, gold_id in zip(predicted_row, gold_row):
            if int(gold_id) == -100:
                continue

            predicted_tags.append(
                id2label[int(predicted_id)]
            )
            gold_tags.append(
                id2label[int(gold_id)]
            )

        predicted_sequences.append(predicted_tags)
        gold_sequences.append(gold_tags)

    return predicted_sequences, gold_sequences


def compute_bert_metrics(eval_prediction):
    predicted_sequences, gold_sequences = (
        extract_bert_tag_sequences(
            eval_prediction.predictions,
            eval_prediction.label_ids,
        )
    )

    metrics = strict_span_metrics(
        predicted_sequences,
        gold_sequences,
    )

    return {
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
    }


# Preserve the validated WNUT BERT training configuration.
training_kwargs = {
    "output_dir": str(BERT_DIR / "checkpoints"),
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "save_total_limit": 2,
    "logging_strategy": "epoch",
    "seed": SEED,
    "data_seed": SEED,
    "report_to": "none",
    "fp16": torch.cuda.is_available(),
}

# Transformers renamed evaluation_strategy to eval_strategy;
# support either version without changing the experiment.
if "eval_strategy" in inspect.signature(
    TrainingArguments.__init__
).parameters:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

bert_training_args = TrainingArguments(
    **training_kwargs
)

trainer_kwargs = {
    "model": bert_model,
    "args": bert_training_args,
    "train_dataset": bert_train_tokenized,
    "eval_dataset": bert_development_tokenized,
    "data_collator": bert_data_collator,
    "compute_metrics": compute_bert_metrics,
}

# Support current and older Transformers Trainer interfaces.
trainer_signature = inspect.signature(Trainer.__init__)

if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = bert_tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = bert_tokenizer

bert_trainer = Trainer(**trainer_kwargs)

bert_trainer.train()

# Save only the selected development-F1 checkpoint needed downstream.
bert_trainer.save_model(str(BERT_MODEL_DIR))
bert_tokenizer.save_pretrained(BERT_MODEL_DIR)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.687765,0.523436,0.593458,0.300948,0.399371
2,0.256132,0.470302,0.657692,0.405213,0.501466
3,0.184727,0.466642,0.596825,0.445498,0.510176


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/09_WNUT17_External_Replication/bert/final_model/tokenizer_config.json',
 '/kaggle/working/09_WNUT17_External_Replication/bert/final_model/tokenizer.json')

### Development performance

In [10]:
# Run the selected BERT checkpoint once on the development population.
bert_development_output = bert_trainer.predict(
    bert_development_tokenized
)

(
    bert_predicted_tags,
    bert_gold_tags,
) = extract_bert_tag_sequences(
    bert_development_output.predictions,
    bert_development_output.label_ids,
)

bert_predicted_sets = [
    span_set_from_tags(tags)
    for tags in bert_predicted_tags
]

bert_gold_sets = [
    span_set_from_tags(tags)
    for tags in bert_gold_tags
]


def metrics_from_span_sets(predicted_sets, gold_sets):
    """Calculate exact typed-span metrics from sentence-level span sets."""
    tp = sum(
        len(predicted & gold)
        for predicted, gold in zip(
            predicted_sets,
            gold_sets,
        )
    )

    fp = sum(
        len(predicted - gold)
        for predicted, gold in zip(
            predicted_sets,
            gold_sets,
        )
    )

    fn = sum(
        len(gold - predicted)
        for predicted, gold in zip(
            predicted_sets,
            gold_sets,
        )
    )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


bert_native_metrics = metrics_from_span_sets(
    bert_predicted_sets,
    bert_gold_sets,
)

# Restrict both predictions and gold spans to SpanNER's width-4 support.
bert_width4_predicted_sets = [
    {
        span
        for span in sentence_spans
        if span[1] - span[0] + 1 <= MAX_SPAN_WIDTH
    }
    for sentence_spans in bert_predicted_sets
]

bert_width4_gold_sets = [
    {
        span
        for span in sentence_spans
        if span[1] - span[0] + 1 <= MAX_SPAN_WIDTH
    }
    for sentence_spans in bert_gold_sets
]

bert_width4_metrics = metrics_from_span_sets(
    bert_width4_predicted_sets,
    bert_width4_gold_sets,
)

bert_development_results = pd.DataFrame([
    {
        "Support": "Native",
        "Precision": bert_native_metrics["precision"],
        "Recall": bert_native_metrics["recall"],
        "F1": bert_native_metrics["f1"],
        "TP": bert_native_metrics["tp"],
        "FP": bert_native_metrics["fp"],
        "FN": bert_native_metrics["fn"],
    },
    {
        "Support": "Width ≤4",
        "Precision": bert_width4_metrics["precision"],
        "Recall": bert_width4_metrics["recall"],
        "F1": bert_width4_metrics["f1"],
        "TP": bert_width4_metrics["tp"],
        "FP": bert_width4_metrics["fp"],
        "FN": bert_width4_metrics["fn"],
    },
])

display(bert_development_results.round(6))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


,Support,Precision,Recall,F1,TP,FP,FN
0,Native,0.596825,0.445498,0.510176,188,127,234
1,Width ≤4,0.596825,0.449761,0.512960,188,127,230


## 5. SpanNER Baseline

The SpanNER-style model enumerates contiguous spans of up to four original words and classifies each candidate as one of the six native WNUT entity types or `NONE`.

Each candidate combines BERT start- and end-boundary representations with tokenised-width and original-word-length embeddings, producing the same 1,686-dimensional span representation used in the primary experiment. `NONE` candidates receive a training weight of 0.5.

The model is trained for at most five epochs using AdamW and a linear OneCycle learning-rate schedule. The checkpoint with the highest strict typed-span F1 on the development partition is retained, with early stopping after two consecutive non-improving epochs. Decoding greedily selects a deterministic non-overlapping set of predicted entities.


### Candidate preparation, lazy dataset and original collator

In [11]:
# Preserve the naming used by the validated WNUT SpanNER implementation.
wnut_train = train_data
wnut_validation = development_data

SPANNER_LABEL_NAMES = [
    "NONE",
    *entity_types,
]

spanner_id2label = {
    index: label
    for index, label in enumerate(SPANNER_LABEL_NAMES)
}

spanner_label2id = {
    label: index
    for index, label in spanner_id2label.items()
}

TOKENISED_WIDTH_BUCKETS = 4
TOKENISED_WIDTH_EMBEDDING_DIM = 50
WORD_LENGTH_EMBEDDING_DIM = 100
SPAN_CLASSIFIER_DROPOUT = 0.2
NEGATIVE_SPAN_WEIGHT = 0.5

SPANNER_CHECKPOINT_DIR = SPANNER_DIR / "checkpoints"
SPANNER_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

spanner_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    use_fast=True,
)

# Reuse the gold spans already established in Section 3.
spanner_gold_by_sentence = {}

for split_name, section_name in [
    ("train", "Train"),
    ("validation", "Development"),
]:
    for sentence_index, sentence_spans in enumerate(
        gold_spans[section_name]
    ):
        spanner_gold_by_sentence[
            (split_name, sentence_index)
        ] = [
            {
                "start": int(span["start"]),
                "end": int(span["end"]),
                "type": str(span["type"]),
                "text": str(span["text"]),
            }
            for span in sentence_spans
        ]


def generate_candidate_spans(tokens, sentence_gold_spans):
    """Enumerate all contiguous width≤4 candidates in original word space."""
    gold_lookup = {
        (
            int(span["start"]),
            int(span["end"]),
        ): str(span["type"])
        for span in sentence_gold_spans
    }

    candidates = []
    candidate_index = 0

    for start in range(len(tokens)):
        maximum_end = min(
            len(tokens) - 1,
            start + MAX_SPAN_WIDTH - 1,
        )

        for end in range(start, maximum_end + 1):
            gold_label = gold_lookup.get(
                (start, end),
                "NONE",
            )

            candidates.append({
                "candidate_index": candidate_index,
                "start": start,
                "end": end,
                "width": end - start + 1,
                "text": " ".join(tokens[start:end + 1]),
                "gold_label": gold_label,
                "gold_label_id": spanner_label2id[gold_label],
            })

            candidate_index += 1

    return candidates


def get_word_subword_boundaries(tokens):
    """Map each original word to its first and last BERT positions."""
    encoded = spanner_tokenizer(
        list(tokens),
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    first_positions = {}
    last_positions = {}

    for subword_index, word_id in enumerate(
        encoded.word_ids()
    ):
        if word_id is None:
            continue

        first_positions.setdefault(
            int(word_id),
            subword_index,
        )

        last_positions[int(word_id)] = (
            subword_index
        )

    missing_words = [
        word_index
        for word_index in range(len(tokens))
        if (
            word_index not in first_positions
            or word_index not in last_positions
        )
    ]

    if missing_words:
        raise RuntimeError(
            "A WNUT sentence was truncated or failed "
            f"word alignment: {missing_words[:10]}"
        )

    return (
        encoded,
        [
            first_positions[index]
            for index in range(len(tokens))
        ],
        [
            last_positions[index]
            for index in range(len(tokens))
        ],
    )


def bucket_tokenised_width(
    first_subword_position,
    last_subword_position,
):
    """Bucket the distance between candidate endpoint subwords."""
    subword_distance = (
        last_subword_position
        - first_subword_position
    )

    return min(
        subword_distance,
        TOKENISED_WIDTH_BUCKETS - 1,
    )


def prepare_spanner_sentence(
    split_name,
    sentence_index,
):
    """Prepare one sentence and its complete SpanNER candidate universe."""
    split_data = {
        "train": wnut_train,
        "validation": wnut_validation,
    }[split_name]

    example = split_data[sentence_index]
    tokens = list(example["tokens"])

    (
        encoded,
        first_positions,
        last_positions,
    ) = get_word_subword_boundaries(tokens)

    sentence_gold_spans = (
        spanner_gold_by_sentence[
            (split_name, sentence_index)
        ]
    )

    candidates = generate_candidate_spans(
        tokens,
        sentence_gold_spans,
    )

    span_word_indices = []
    span_subword_indices = []
    span_word_lengths = []
    tokenised_width_buckets = []
    labels = []
    span_weights = []
    candidate_texts = []

    for candidate in candidates:
        start = candidate["start"]
        end = candidate["end"]

        start_subword = first_positions[start]
        end_subword = last_positions[end]

        span_word_indices.append(
            [start, end]
        )

        span_subword_indices.append(
            [start_subword, end_subword]
        )

        span_word_lengths.append(
            end - start + 1
        )

        tokenised_width_buckets.append(
            bucket_tokenised_width(
                start_subword,
                end_subword,
            )
        )

        label_id = candidate["gold_label_id"]
        labels.append(label_id)

        span_weights.append(
            NEGATIVE_SPAN_WEIGHT
            if label_id == spanner_label2id["NONE"]
            else 1.0
        )

        candidate_texts.append(
            candidate["text"]
        )

    return {
        "input_ids": torch.tensor(
            encoded["input_ids"],
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            encoded["attention_mask"],
            dtype=torch.long,
        ),
        "token_type_ids": torch.tensor(
            encoded.get(
                "token_type_ids",
                [0] * len(encoded["input_ids"]),
            ),
            dtype=torch.long,
        ),
        "span_word_indices": torch.tensor(
            span_word_indices,
            dtype=torch.long,
        ),
        "span_subword_indices": torch.tensor(
            span_subword_indices,
            dtype=torch.long,
        ),
        "span_word_lengths": torch.tensor(
            span_word_lengths,
            dtype=torch.long,
        ),
        "tokenised_width_buckets": torch.tensor(
            tokenised_width_buckets,
            dtype=torch.long,
        ),
        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),
        "span_weights": torch.tensor(
            span_weights,
            dtype=torch.float,
        ),
        "span_mask": torch.ones(
            len(candidates),
            dtype=torch.bool,
        ),
        "split": split_name,
        "sentence_index": sentence_index,
        "tokens": tokens,
        "candidate_texts": candidate_texts,
    }


class SpanNERSentenceDataset(TorchDataset):
    """Prepare SpanNER sentences lazily, matching the original WNUT run."""

    def __init__(self, split_name):
        self.split_name = split_name
        self.dataset = {
            "train": wnut_train,
            "validation": wnut_validation,
        }[split_name]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, sentence_index):
        return prepare_spanner_sentence(
            self.split_name,
            int(sentence_index),
        )


train_span_dataset = SpanNERSentenceDataset(
    "train"
)

validation_span_dataset = SpanNERSentenceDataset(
    "validation"
)


def collate_spanner_batch(batch):
    """Pad BERT-token and candidate dimensions within one batch."""
    batch_size = len(batch)

    maximum_sequence_length = max(
        len(sentence["input_ids"])
        for sentence in batch
    )

    maximum_candidate_count = max(
        len(sentence["labels"])
        for sentence in batch
    )

    input_ids = torch.full(
        (
            batch_size,
            maximum_sequence_length,
        ),
        fill_value=spanner_tokenizer.pad_token_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (
            batch_size,
            maximum_sequence_length,
        ),
        dtype=torch.long,
    )

    token_type_ids = torch.zeros(
        (
            batch_size,
            maximum_sequence_length,
        ),
        dtype=torch.long,
    )

    span_word_indices = torch.zeros(
        (
            batch_size,
            maximum_candidate_count,
            2,
        ),
        dtype=torch.long,
    )

    span_subword_indices = torch.zeros(
        (
            batch_size,
            maximum_candidate_count,
            2,
        ),
        dtype=torch.long,
    )

    span_word_lengths = torch.zeros(
        (
            batch_size,
            maximum_candidate_count,
        ),
        dtype=torch.long,
    )

    tokenised_width_buckets = torch.zeros(
        (
            batch_size,
            maximum_candidate_count,
        ),
        dtype=torch.long,
    )

    labels = torch.full(
        (
            batch_size,
            maximum_candidate_count,
        ),
        fill_value=-100,
        dtype=torch.long,
    )

    span_weights = torch.zeros(
        (
            batch_size,
            maximum_candidate_count,
        ),
        dtype=torch.float,
    )

    span_mask = torch.zeros(
        (
            batch_size,
            maximum_candidate_count,
        ),
        dtype=torch.bool,
    )

    metadata = []

    for batch_index, sentence in enumerate(batch):
        sequence_length = len(
            sentence["input_ids"]
        )

        candidate_count = len(
            sentence["labels"]
        )

        input_ids[
            batch_index,
            :sequence_length,
        ] = sentence["input_ids"]

        attention_mask[
            batch_index,
            :sequence_length,
        ] = sentence["attention_mask"]

        token_type_ids[
            batch_index,
            :sequence_length,
        ] = sentence["token_type_ids"]

        span_word_indices[
            batch_index,
            :candidate_count,
        ] = sentence["span_word_indices"]

        span_subword_indices[
            batch_index,
            :candidate_count,
        ] = sentence["span_subword_indices"]

        span_word_lengths[
            batch_index,
            :candidate_count,
        ] = sentence["span_word_lengths"]

        tokenised_width_buckets[
            batch_index,
            :candidate_count,
        ] = sentence["tokenised_width_buckets"]

        labels[
            batch_index,
            :candidate_count,
        ] = sentence["labels"]

        span_weights[
            batch_index,
            :candidate_count,
        ] = sentence["span_weights"]

        span_mask[
            batch_index,
            :candidate_count,
        ] = True

        metadata.append({
            "split": sentence["split"],
            "sentence_index": sentence["sentence_index"],
            "tokens": sentence["tokens"],
            "candidate_texts": sentence["candidate_texts"],
            "candidate_count": candidate_count,
        })

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
        "span_word_indices": span_word_indices,
        "span_subword_indices": span_subword_indices,
        "span_word_lengths": span_word_lengths,
        "tokenised_width_buckets": tokenised_width_buckets,
        "labels": labels,
        "span_weights": span_weights,
        "span_mask": span_mask,
        "metadata": metadata,
    }


TRAIN_BATCH_SIZE = 10
EVAL_BATCH_SIZE = 10

dataloader_generator = torch.Generator()
dataloader_generator.manual_seed(SEED)

train_span_dataloader = DataLoader(
    train_span_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_spanner_batch,
    generator=dataloader_generator,
)

validation_span_dataloader = DataLoader(
    validation_span_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_spanner_batch,
)

# The original WNUT implementation reset this generator
# before model training began.
dataloader_generator.manual_seed(SEED)

print(
    "SpanNER training batches:",
    len(train_span_dataloader),
)

print(
    "SpanNER validation batches:",
    len(validation_span_dataloader),
)

SpanNER training batches: 340
SpanNER validation batches: 51


### Original architecture, loss and decoder

In [12]:
class SpanClassifier(nn.Module):
    """Two-layer nonlinear classifier over candidate-span representations."""

    def __init__(
        self,
        input_dim,
        hidden_dim,
        number_of_labels,
        dropout_rate,
    ):
        super().__init__()

        self.linear_1 = nn.Linear(
            input_dim,
            hidden_dim,
        )

        self.dropout = nn.Dropout(
            dropout_rate
        )

        self.linear_2 = nn.Linear(
            hidden_dim,
            number_of_labels,
        )

    def forward(
        self,
        span_representations,
    ):
        hidden = self.linear_1(
            span_representations
        )

        hidden = F.gelu(hidden)
        hidden = self.dropout(hidden)

        return self.linear_2(hidden)


class BERTSpanNER(nn.Module):
    """BERT endpoint-based SpanNER model used in the WNUT experiment."""

    def __init__(
        self,
        model_checkpoint,
        maximum_span_width,
        number_of_width_buckets,
        width_embedding_dim,
        word_length_embedding_dim,
        number_of_labels,
        classifier_dropout,
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_checkpoint
        )

        self.hidden_size = int(
            self.bert.config.hidden_size
        )

        self.tokenised_width_embedding = nn.Embedding(
            number_of_width_buckets,
            width_embedding_dim,
        )

        self.word_length_embedding = nn.Embedding(
            maximum_span_width + 1,
            word_length_embedding_dim,
            padding_idx=0,
        )

        self.span_representation_dim = (
            2 * self.hidden_size
            + width_embedding_dim
            + word_length_embedding_dim
        )

        self.classifier = SpanClassifier(
            input_dim=self.span_representation_dim,
            hidden_dim=self.span_representation_dim,
            number_of_labels=number_of_labels,
            dropout_rate=classifier_dropout,
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        span_subword_indices,
        span_word_lengths,
        tokenised_width_buckets,
    ):
        sequence_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        ).last_hidden_state

        start_indices = (
            span_subword_indices[:, :, 0]
        )

        end_indices = (
            span_subword_indices[:, :, 1]
        )

        hidden_size = sequence_output.shape[-1]

        start_vectors = torch.gather(
            sequence_output,
            dim=1,
            index=(
                start_indices
                .unsqueeze(-1)
                .expand(
                    -1,
                    -1,
                    hidden_size,
                )
            ),
        )

        end_vectors = torch.gather(
            sequence_output,
            dim=1,
            index=(
                end_indices
                .unsqueeze(-1)
                .expand(
                    -1,
                    -1,
                    hidden_size,
                )
            ),
        )

        width_vectors = (
            self.tokenised_width_embedding(
                tokenised_width_buckets
            )
        )

        word_length_vectors = F.relu(
            self.word_length_embedding(
                span_word_lengths
            )
        )

        span_representations = torch.cat(
            [
                start_vectors,
                end_vectors,
                width_vectors,
                word_length_vectors,
            ],
            dim=-1,
        )

        return self.classifier(
            span_representations
        )


# Model construction occurs here, after the DataLoaders,
# matching the ordering of the validated WNUT notebook.
spanner_model = BERTSpanNER(
    model_checkpoint=MODEL_CHECKPOINT,
    maximum_span_width=MAX_SPAN_WIDTH,
    number_of_width_buckets=TOKENISED_WIDTH_BUCKETS,
    width_embedding_dim=TOKENISED_WIDTH_EMBEDDING_DIM,
    word_length_embedding_dim=WORD_LENGTH_EMBEDDING_DIM,
    number_of_labels=len(SPANNER_LABEL_NAMES),
    classifier_dropout=SPAN_CLASSIFIER_DROPOUT,
).to(DEVICE)

assert (
    spanner_model.span_representation_dim
    == 1686
)


def compute_span_loss(
    logits,
    labels,
    span_mask,
    span_weights=None,
    apply_span_weights=False,
):
    """Candidate-level cross-entropy used by the original WNUT run."""
    number_of_labels = logits.shape[-1]

    per_candidate_loss = F.cross_entropy(
        logits.view(
            -1,
            number_of_labels,
        ),
        labels.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view_as(labels)

    if apply_span_weights:
        if span_weights is None:
            raise ValueError(
                "Span weights are required."
            )

        per_candidate_loss = (
            per_candidate_loss
            * span_weights
        )

    real_losses = torch.masked_select(
        per_candidate_loss,
        span_mask.bool(),
    )

    return real_losses.mean()


def spans_overlap(first_span, second_span):
    return not (
        first_span["end"] < second_span["start"]
        or second_span["end"] < first_span["start"]
    )


def decode_non_overlapping_spans(
    probabilities,
    span_word_indices,
    span_mask,
):
    """Apply the frozen deterministic SpanNER decoder."""
    ranked_candidates = []

    real_candidate_indices = (
        torch.nonzero(
            span_mask.bool(),
            as_tuple=False,
        )
        .flatten()
        .tolist()
    )

    for candidate_index in real_candidate_indices:
        predicted_label_id = int(
            probabilities[
                candidate_index
            ].argmax().item()
        )

        if (
            predicted_label_id
            == spanner_label2id["NONE"]
        ):
            continue

        start, end = (
            span_word_indices[
                candidate_index
            ]
            .detach()
            .cpu()
            .tolist()
        )

        confidence = float(
            probabilities[
                candidate_index,
                predicted_label_id,
            ].item()
        )

        ranked_candidates.append({
            "candidate_index": candidate_index,
            "start": int(start),
            "end": int(end),
            "width": int(end - start + 1),
            "type": spanner_id2label[
                predicted_label_id
            ],
            "confidence": confidence,
        })

    ranked_candidates.sort(
        key=lambda candidate: (
            -candidate["confidence"],
            -candidate["width"],
            candidate["start"],
            candidate["end"],
            candidate["candidate_index"],
        )
    )

    selected = []

    for candidate in ranked_candidates:
        if any(
            spans_overlap(
                candidate,
                previous,
            )
            for previous in selected
        ):
            continue

        selected.append(candidate)

    selected.sort(
        key=lambda candidate: (
            candidate["start"],
            candidate["end"],
            candidate["type"],
        )
    )

    return selected


def move_span_batch_to_device(batch):
    """Move model inputs and supervision tensors to the active device."""
    model_inputs = {
        "input_ids": batch[
            "input_ids"
        ].to(DEVICE),
        "attention_mask": batch[
            "attention_mask"
        ].to(DEVICE),
        "token_type_ids": batch[
            "token_type_ids"
        ].to(DEVICE),
        "span_subword_indices": batch[
            "span_subword_indices"
        ].to(DEVICE),
        "span_word_lengths": batch[
            "span_word_lengths"
        ].to(DEVICE),
        "tokenised_width_buckets": batch[
            "tokenised_width_buckets"
        ].to(DEVICE),
    }

    return (
        model_inputs,
        batch["labels"].to(DEVICE),
        batch["span_mask"].to(DEVICE),
        batch["span_weights"].to(DEVICE),
    )


def evaluate_spanner_model(
    model,
    dataloader,
    split_name,
    return_predictions=False,
):
    """Evaluate strict typed-span performance using the frozen decoder."""
    model.eval()

    total_loss = 0.0
    total_candidates = 0
    correct_candidates = 0

    predicted_sets = []
    gold_sets = []

    with torch.no_grad():
        for batch in dataloader:
            (
                model_inputs,
                labels,
                span_mask,
                _,
            ) = move_span_batch_to_device(
                batch
            )

            logits = model(**model_inputs)

            probabilities = torch.softmax(
                logits,
                dim=-1,
            )

            loss = compute_span_loss(
                logits=logits,
                labels=labels,
                span_mask=span_mask,
                apply_span_weights=False,
            )

            real_candidate_count = int(
                span_mask.sum().item()
            )

            total_loss += (
                loss.item()
                * real_candidate_count
            )

            total_candidates += (
                real_candidate_count
            )

            predicted_ids = probabilities.argmax(
                dim=-1
            )

            correct_candidates += int(
                (
                    predicted_ids.eq(labels)
                    & span_mask.bool()
                )
                .sum()
                .item()
            )

            for sentence_position, metadata in enumerate(
                batch["metadata"]
            ):
                candidate_count = int(
                    batch["span_mask"][
                        sentence_position
                    ].sum().item()
                )

                sentence_probabilities = (
                    probabilities[
                        sentence_position,
                        :candidate_count,
                    ]
                    .detach()
                    .cpu()
                )

                sentence_boundaries = (
                    batch["span_word_indices"][
                        sentence_position,
                        :candidate_count,
                    ]
                )

                sentence_mask = torch.ones(
                    candidate_count,
                    dtype=torch.bool,
                )

                decoded = decode_non_overlapping_spans(
                    probabilities=sentence_probabilities,
                    span_word_indices=sentence_boundaries,
                    span_mask=sentence_mask,
                )

                predicted_sets.append({
                    (
                        span["start"],
                        span["end"],
                        span["type"],
                    )
                    for span in decoded
                })

                sentence_index = int(
                    metadata["sentence_index"]
                )

                gold_sets.append({
                    (
                        int(span["start"]),
                        int(span["end"]),
                        str(span["type"]),
                    )
                    for span in (
                        spanner_gold_by_sentence[
                            (
                                split_name,
                                sentence_index,
                            )
                        ]
                    )
                })

    metrics = metrics_from_span_sets(
        predicted_sets,
        gold_sets,
    )

    metrics["loss"] = (
        total_loss / total_candidates
    )

    metrics["candidate_accuracy"] = (
        correct_candidates
        / total_candidates
    )

    if return_predictions:
        return (
            metrics,
            predicted_sets,
            gold_sets,
        )

    return metrics


print(
    "Span representation dimension:",
    spanner_model.span_representation_dim,
)

print(
    "Output classes:",
    SPANNER_LABEL_NAMES,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Span representation dimension: 1686
Output classes: ['NONE', 'corporation', 'creative-work', 'group', 'location', 'person', 'product']


### Original training loop and baseline comparison

In [13]:
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
ADAM_BETAS = (0.9, 0.98)
ADAM_EPSILON = 1e-8
MAX_GRAD_NORM = 1.0

MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
MINIMUM_F1_IMPROVEMENT = 1e-6

USE_MIXED_PRECISION = (
    DEVICE.type == "cuda"
)

BEST_SPANNER_CHECKPOINT = (
    SPANNER_CHECKPOINT_DIR
    / "best_validation_f1.pt"
)

no_decay_terms = (
    "bias",
    "LayerNorm.weight",
)

optimizer_parameter_groups = [
    {
        "params": [
            parameter
            for name, parameter
            in spanner_model.named_parameters()
            if not any(
                term in name
                for term in no_decay_terms
            )
        ],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [
            parameter
            for name, parameter
            in spanner_model.named_parameters()
            if any(
                term in name
                for term in no_decay_terms
            )
        ],
        "weight_decay": 0.0,
    },
]

optimizer = torch.optim.AdamW(
    optimizer_parameter_groups,
    lr=LEARNING_RATE,
    betas=ADAM_BETAS,
    eps=ADAM_EPSILON,
)

TOTAL_PLANNED_STEPS = (
    len(train_span_dataloader)
    * MAX_EPOCHS
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=TOTAL_PLANNED_STEPS,
    pct_start=0.0,
    anneal_strategy="linear",
    final_div_factor=1e4,
)

gradient_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_MIXED_PRECISION,
)

# Frozen training shuffle.
dataloader_generator.manual_seed(SEED)

best_validation_f1 = -float("inf")
best_epoch = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    spanner_model.train()

    progress = tqdm(
        train_span_dataloader,
        desc=(
            f"SpanNER epoch "
            f"{epoch}/{MAX_EPOCHS}"
        ),
    )

    for batch in progress:
        (
            model_inputs,
            labels,
            span_mask,
            span_weights,
        ) = move_span_batch_to_device(
            batch
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_MIXED_PRECISION,
        ):
            logits = spanner_model(
                **model_inputs
            )

            training_loss = compute_span_loss(
                logits=logits,
                labels=labels,
                span_mask=span_mask,
                span_weights=span_weights,
                apply_span_weights=True,
            )

        if not torch.isfinite(
            training_loss
        ).item():
            raise RuntimeError(
                "Non-finite SpanNER training loss."
            )

        gradient_scaler.scale(
            training_loss
        ).backward()

        gradient_scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            spanner_model.parameters(),
            max_norm=MAX_GRAD_NORM,
        )

        scale_before_step = (
            gradient_scaler.get_scale()
        )

        gradient_scaler.step(
            optimizer
        )

        gradient_scaler.update()

        scale_after_step = (
            gradient_scaler.get_scale()
        )

        optimizer_step_skipped = (
            scale_after_step
            < scale_before_step
        )

        if not optimizer_step_skipped:
            scheduler.step()

        progress.set_postfix(
            loss=f"{training_loss.item():.4f}"
        )

    validation_metrics = (
        evaluate_spanner_model(
            model=spanner_model,
            dataloader=validation_span_dataloader,
            split_name="validation",
        )
    )

    current_f1 = float(
        validation_metrics["f1"]
    )

    print(
        f"Epoch {epoch}: "
        f"P={validation_metrics['precision']:.4f} "
        f"R={validation_metrics['recall']:.4f} "
        f"F1={current_f1:.4f}"
    )

    if (
        current_f1
        > best_validation_f1
        + MINIMUM_F1_IMPROVEMENT
    ):
        best_validation_f1 = (
            current_f1
        )

        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": {
                    key: value.detach().cpu()
                    for key, value
                    in spanner_model
                    .state_dict()
                    .items()
                },
                "validation_metrics": (
                    validation_metrics
                ),
            },
            BEST_SPANNER_CHECKPOINT,
        )

    else:
        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            "SpanNER early stopping "
            f"after epoch {epoch}."
        )
        break


# Restore the development-selected checkpoint.
best_checkpoint = torch.load(
    BEST_SPANNER_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False,
)

spanner_model.load_state_dict(
    best_checkpoint["model_state_dict"],
    strict=True,
)

spanner_model.to(DEVICE)
spanner_model.eval()

(
    spanner_native_metrics,
    spanner_validation_predicted_sets,
    spanner_validation_gold_sets,
) = evaluate_spanner_model(
    model=spanner_model,
    dataloader=validation_span_dataloader,
    split_name="validation",
    return_predictions=True,
)

# Symmetric width≤4 comparison.
spanner_width4_predicted_sets = [
    {
        span
        for span in sentence_spans
        if (
            span[1] - span[0] + 1
            <= MAX_SPAN_WIDTH
        )
    }
    for sentence_spans
    in spanner_validation_predicted_sets
]

spanner_width4_gold_sets = [
    {
        span
        for span in sentence_spans
        if (
            span[1] - span[0] + 1
            <= MAX_SPAN_WIDTH
        )
    }
    for sentence_spans
    in spanner_validation_gold_sets
]

spanner_width4_metrics = metrics_from_span_sets(
    spanner_width4_predicted_sets,
    spanner_width4_gold_sets,
)

baseline_comparison = pd.DataFrame([
    {
        "Model": "BERT",
        "Support": "Native",
        "Precision": bert_native_metrics["precision"],
        "Recall": bert_native_metrics["recall"],
        "F1": bert_native_metrics["f1"],
        "TP": bert_native_metrics["tp"],
        "FP": bert_native_metrics["fp"],
        "FN": bert_native_metrics["fn"],
    },
    {
        "Model": "SpanNER",
        "Support": "Native",
        "Precision": spanner_native_metrics["precision"],
        "Recall": spanner_native_metrics["recall"],
        "F1": spanner_native_metrics["f1"],
        "TP": spanner_native_metrics["tp"],
        "FP": spanner_native_metrics["fp"],
        "FN": spanner_native_metrics["fn"],
    },
    {
        "Model": "BERT",
        "Support": "Width ≤4",
        "Precision": bert_width4_metrics["precision"],
        "Recall": bert_width4_metrics["recall"],
        "F1": bert_width4_metrics["f1"],
        "TP": bert_width4_metrics["tp"],
        "FP": bert_width4_metrics["fp"],
        "FN": bert_width4_metrics["fn"],
    },
    {
        "Model": "SpanNER",
        "Support": "Width ≤4",
        "Precision": spanner_width4_metrics["precision"],
        "Recall": spanner_width4_metrics["recall"],
        "F1": spanner_width4_metrics["f1"],
        "TP": spanner_width4_metrics["tp"],
        "FP": spanner_width4_metrics["fp"],
        "FN": spanner_width4_metrics["fn"],
    },
])

print(
    "Best SpanNER epoch:",
    best_epoch,
)

display(
    baseline_comparison.round(6)
)

# Save only the selected model needed by the remaining notebook.
torch.save(
    {
        "model_state_dict": {
            key: value.detach().cpu()
            for key, value
            in spanner_model
            .state_dict()
            .items()
        },
        "span_label_names": (
            SPANNER_LABEL_NAMES
        ),
        "best_epoch": best_epoch,
        "maximum_span_width": (
            MAX_SPAN_WIDTH
        ),
    },
    SPANNER_MODEL_DIR
    / "spanner_model.pt",
)

spanner_tokenizer.save_pretrained(
    SPANNER_MODEL_DIR
)

SpanNER epoch 1/5:   0%|          | 0/340 [00:00<?, ?it/s]

Epoch 1: P=0.9041 R=0.1564 F1=0.2667


SpanNER epoch 2/5:   0%|          | 0/340 [00:00<?, ?it/s]

Epoch 2: P=0.7214 R=0.4479 F1=0.5526


SpanNER epoch 3/5:   0%|          | 0/340 [00:00<?, ?it/s]

Epoch 3: P=0.7251 R=0.4313 F1=0.5409


SpanNER epoch 4/5:   0%|          | 0/340 [00:00<?, ?it/s]

Epoch 4: P=0.7186 R=0.4479 F1=0.5518
SpanNER early stopping after epoch 4.
Best SpanNER epoch: 2


,Model,Support,Precision,Recall,F1,TP,FP,FN
0,BERT,Native,0.596825,0.445498,0.510176,188,127,234
1,SpanNER,Native,0.721374,0.447867,0.552632,189,73,233
2,BERT,Width ≤4,0.596825,0.449761,0.512960,188,127,230
3,SpanNER,Width ≤4,0.721374,0.452153,0.555882,189,73,229


('/kaggle/working/09_WNUT17_External_Replication/spanner/final_model/tokenizer_config.json',
 '/kaggle/working/09_WNUT17_External_Replication/spanner/final_model/tokenizer.json')

## 6. Temperature Scaling

After checkpoint selection, both NER models are frozen. Deterministic logits are generated for the development, probability-calibration and conformal-calibration populations.

A single positive temperature is fitted separately for each architecture using only the 303 probability-calibration sentences:

* BERT uses original-word BIO logits;
* SpanNER uses logits from the complete width≤4 candidate population, including `NONE`.

The fitted temperature is retained only when it reduces multiclass NLL on the independent development partition; otherwise, the unscaled value T=1 is used. The retained temperatures are subsequently applied to boundary-uncertainty construction and conformal prediction.

No binary boundary calibrator is fitted in the WNUT replication.


In [14]:
# Use concise aliases matching the remaining WNUT workflow.
wnut_validation = development_data
wnut_probability_calibration = probability_calibration_data
wnut_conformal_calibration = conformal_calibration_data


def bert_word_logits_for_split(
    dataset_split,
    description,
):
    """Return one frozen BERT logit vector per original word."""
    model = bert_trainer.model.to(DEVICE)
    model.eval()

    outputs = []

    with torch.no_grad():
        for sentence_id, example in enumerate(
            tqdm(dataset_split, desc=description)
        ):
            tokens = list(example["tokens"])

            encoded = bert_tokenizer(
                tokens,
                is_split_into_words=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )

            word_ids = encoded.word_ids(
                batch_index=0
            )

            # Retain the first subword representation for each word.
            first_subword = {}

            for position, word_id in enumerate(word_ids):
                if word_id is None:
                    continue

                first_subword.setdefault(
                    int(word_id),
                    position,
                )

            if len(first_subword) != len(tokens):
                raise RuntimeError(
                    f"BERT truncation affected sentence {sentence_id}."
                )

            model_inputs = {
                key: value.to(DEVICE)
                for key, value in encoded.items()
            }

            logits = (
                model(**model_inputs)
                .logits[0]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float64)
            )

            word_logits = logits[
                [
                    first_subword[index]
                    for index in range(len(tokens))
                ]
            ]

            outputs.append({
                "sentence_id": sentence_id,
                "tokens": tokens,
                "word_logits": word_logits,
                "gold_label_ids": np.asarray(
                    example["ner_tags"],
                    dtype=np.int64,
                ),
            })

    return outputs


def example_to_gold_spans(example):
    """Convert one WNUT example to the shared typed-span representation."""
    tokens = list(example["tokens"])

    tags = [
        id2label[int(tag_id)]
        for tag_id in example["ner_tags"]
    ]

    return bio_to_spans(
        tokens,
        tags,
    )


def prepare_spanner_example_generic(example):
    """Prepare all width≤4 candidates for frozen SpanNER inference."""
    tokens = list(example["tokens"])

    (
        encoded,
        first_positions,
        last_positions,
    ) = get_word_subword_boundaries(tokens)

    sentence_gold_spans = example_to_gold_spans(
        example
    )

    candidates = generate_candidate_spans(
        tokens,
        sentence_gold_spans,
    )

    span_subword_indices = []
    span_word_lengths = []
    tokenised_width_buckets = []

    for candidate in candidates:
        start = int(candidate["start"])
        end = int(candidate["end"])

        start_subword = first_positions[start]
        end_subword = last_positions[end]

        span_subword_indices.append([
            start_subword,
            end_subword,
        ])

        span_word_lengths.append(
            end - start + 1
        )

        tokenised_width_buckets.append(
            bucket_tokenised_width(
                start_subword,
                end_subword,
            )
        )

    return {
        "tokens": tokens,
        "gold_spans": sentence_gold_spans,
        "candidates": candidates,
        "input_ids": torch.tensor(
            encoded["input_ids"],
            dtype=torch.long,
        ).unsqueeze(0),
        "attention_mask": torch.tensor(
            encoded["attention_mask"],
            dtype=torch.long,
        ).unsqueeze(0),
        "token_type_ids": torch.tensor(
            encoded.get(
                "token_type_ids",
                [0] * len(encoded["input_ids"]),
            ),
            dtype=torch.long,
        ).unsqueeze(0),
        "span_subword_indices": torch.tensor(
            span_subword_indices,
            dtype=torch.long,
        ).unsqueeze(0),
        "span_word_lengths": torch.tensor(
            span_word_lengths,
            dtype=torch.long,
        ).unsqueeze(0),
        "tokenised_width_buckets": torch.tensor(
            tokenised_width_buckets,
            dtype=torch.long,
        ).unsqueeze(0),
    }


def spanner_logits_for_split(
    dataset_split,
    description,
):
    """Run the frozen SpanNER model over one WNUT population."""
    spanner_model.to(DEVICE)
    spanner_model.eval()

    sentence_outputs = []
    candidate_rows = []
    decoded_sets = []
    gold_sets = []

    with torch.no_grad():
        for sentence_id, example in enumerate(
            tqdm(dataset_split, desc=description)
        ):
            prepared = prepare_spanner_example_generic(
                example
            )

            model_inputs = {
                "input_ids":
                    prepared["input_ids"].to(DEVICE),
                "attention_mask":
                    prepared["attention_mask"].to(DEVICE),
                "token_type_ids":
                    prepared["token_type_ids"].to(DEVICE),
                "span_subword_indices":
                    prepared["span_subword_indices"].to(DEVICE),
                "span_word_lengths":
                    prepared["span_word_lengths"].to(DEVICE),
                "tokenised_width_buckets":
                    prepared["tokenised_width_buckets"].to(DEVICE),
            }

            logits = (
                spanner_model(**model_inputs)[0]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float64)
            )

            probabilities = torch.softmax(
                torch.tensor(logits),
                dim=-1,
            )

            span_word_indices = torch.tensor(
                [
                    [
                        candidate["start"],
                        candidate["end"],
                    ]
                    for candidate in prepared["candidates"]
                ],
                dtype=torch.long,
            )

            span_mask = torch.ones(
                len(prepared["candidates"]),
                dtype=torch.bool,
            )

            decoded = decode_non_overlapping_spans(
                probabilities,
                span_word_indices,
                span_mask,
            )

            decoded_sets.append({
                (
                    int(span["start"]),
                    int(span["end"]),
                    str(span["type"]),
                )
                for span in decoded
            })

            gold_sets.append({
                (
                    int(span["start"]),
                    int(span["end"]),
                    str(span["type"]),
                )
                for span in prepared["gold_spans"]
            })

            # Retain complete candidate logits because they are reused
            # for temperature scaling and later conformal construction.
            for candidate_index, (
                candidate,
                candidate_logits,
            ) in enumerate(
                zip(
                    prepared["candidates"],
                    logits,
                )
            ):
                row = {
                    "sentence_id": sentence_id,
                    "candidate_index": candidate_index,
                    "start": int(candidate["start"]),
                    "end": int(candidate["end"]),
                    "width": int(candidate["width"]),
                    "gold_label": str(
                        candidate["gold_label"]
                    ),
                    "gold_label_id": int(
                        candidate["gold_label_id"]
                    ),
                }

                for label_index, label_name in (
                    spanner_id2label.items()
                ):
                    row[
                        f"logit_{label_name}"
                    ] = float(
                        candidate_logits[label_index]
                    )

                candidate_rows.append(row)

            sentence_outputs.append({
                "sentence_id": sentence_id,
                "tokens": prepared["tokens"],
            })

    return {
        "sentences": sentence_outputs,
        "candidate_logits": pd.DataFrame(
            candidate_rows
        ),
        "decoded_sets": decoded_sets,
        "gold_sets": gold_sets,
    }


# Generate frozen logits once and reuse them throughout the notebook.
bert_validation_logits = bert_word_logits_for_split(
    wnut_validation,
    "BERT logits: development",
)

bert_probability_logits = bert_word_logits_for_split(
    wnut_probability_calibration,
    "BERT logits: probability calibration",
)

bert_conformal_logits = bert_word_logits_for_split(
    wnut_conformal_calibration,
    "BERT logits: conformal calibration",
)

spanner_validation_inference = spanner_logits_for_split(
    wnut_validation,
    "SpanNER logits: development",
)

spanner_probability_inference = spanner_logits_for_split(
    wnut_probability_calibration,
    "SpanNER logits: probability calibration",
)

spanner_conformal_inference = spanner_logits_for_split(
    wnut_conformal_calibration,
    "SpanNER logits: conformal calibration",
)

print("Frozen-model inference complete.")

BERT logits: development:   0%|          | 0/505 [00:00<?, ?it/s]

BERT logits: probability calibration:   0%|          | 0/303 [00:00<?, ?it/s]

BERT logits: conformal calibration:   0%|          | 0/201 [00:00<?, ?it/s]

SpanNER logits: development:   0%|          | 0/505 [00:00<?, ?it/s]

SpanNER logits: probability calibration:   0%|          | 0/303 [00:00<?, ?it/s]

SpanNER logits: conformal calibration:   0%|          | 0/201 [00:00<?, ?it/s]

Frozen-model inference complete.


### Fit and retain temperatures

In [15]:
def flatten_bert_logits(sentence_outputs):
    """Combine sentence-level BERT logits into one calibration population."""
    logits = np.concatenate(
        [
            item["word_logits"]
            for item in sentence_outputs
        ],
        axis=0,
    )

    labels = np.concatenate(
        [
            item["gold_label_ids"]
            for item in sentence_outputs
        ],
        axis=0,
    )

    return logits, labels


def flatten_spanner_logits(inference_output):
    """Combine complete SpanNER candidate logits and targets."""
    frame = inference_output[
        "candidate_logits"
    ]

    logit_columns = [
        f"logit_{spanner_id2label[index]}"
        for index in range(
            len(SPANNER_LABEL_NAMES)
        )
    ]

    return (
        frame[logit_columns].to_numpy(
            dtype=np.float64
        ),
        frame["gold_label_id"].to_numpy(
            dtype=np.int64
        ),
    )


def fit_temperature_scalar(
    logits_np,
    labels_np,
):
    """Fit one positive temperature by minimising multiclass NLL."""
    logits = torch.tensor(
        logits_np,
        dtype=torch.float64,
    )

    labels = torch.tensor(
        labels_np,
        dtype=torch.long,
    )

    # Optimising log(T) guarantees a positive temperature.
    log_temperature = torch.nn.Parameter(
        torch.zeros(
            1,
            dtype=torch.float64,
        )
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=100,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad()

        temperature = torch.exp(
            log_temperature
        )

        loss = F.cross_entropy(
            logits / temperature,
            labels,
        )

        loss.backward()

        return loss

    optimizer.step(closure)

    return float(
        torch.exp(
            log_temperature.detach()
        ).item()
    )


def softmax_numpy(logits):
    """Numerically stable row-wise softmax."""
    shifted = (
        logits
        - np.max(
            logits,
            axis=1,
            keepdims=True,
        )
    )

    exponentiated = np.exp(shifted)

    return (
        exponentiated
        / exponentiated.sum(
            axis=1,
            keepdims=True,
        )
    )


def multiclass_nll(
    logits,
    labels,
    temperature=1.0,
):
    """Mean multiclass negative log-likelihood."""
    scaled = logits / float(temperature)

    shifted = (
        scaled
        - np.max(
            scaled,
            axis=1,
            keepdims=True,
        )
    )

    log_sum_exp = np.log(
        np.exp(shifted).sum(axis=1)
    )

    correct_logits = shifted[
        np.arange(len(labels)),
        labels,
    ]

    return float(
        np.mean(
            log_sum_exp - correct_logits
        )
    )


def multiclass_ece(
    logits,
    labels,
    temperature=1.0,
    bins=15,
):
    """Calculate standard confidence-based multiclass ECE."""
    probabilities = softmax_numpy(
        logits / float(temperature)
    )

    confidence = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)

    correctness = (
        predictions == labels
    ).astype(float)

    edges = np.linspace(
        0.0,
        1.0,
        bins + 1,
    )

    ece = 0.0

    for index in range(bins):
        lower = edges[index]
        upper = edges[index + 1]

        if index == bins - 1:
            mask = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                & (confidence < upper)
            )

        if not mask.any():
            continue

        ece += mask.mean() * abs(
            correctness[mask].mean()
            - confidence[mask].mean()
        )

    return float(ece)


# Calibration labels are used only from the dedicated
# probability-calibration population.
bert_probability_x, bert_probability_y = (
    flatten_bert_logits(
        bert_probability_logits
    )
)

spanner_probability_x, spanner_probability_y = (
    flatten_spanner_logits(
        spanner_probability_inference
    )
)

# Development data are used only to decide whether the fitted
# temperature should be retained.
bert_validation_x, bert_validation_y = (
    flatten_bert_logits(
        bert_validation_logits
    )
)

spanner_validation_x, spanner_validation_y = (
    flatten_spanner_logits(
        spanner_validation_inference
    )
)

bert_temperature_fitted = fit_temperature_scalar(
    bert_probability_x,
    bert_probability_y,
)

spanner_temperature_fitted = fit_temperature_scalar(
    spanner_probability_x,
    spanner_probability_y,
)

temperature_rows = []

for (
    model_name,
    validation_x,
    validation_y,
    fitted_temperature,
) in [
    (
        "BERT",
        bert_validation_x,
        bert_validation_y,
        bert_temperature_fitted,
    ),
    (
        "SpanNER",
        spanner_validation_x,
        spanner_validation_y,
        spanner_temperature_fitted,
    ),
]:
    nll_before = multiclass_nll(
        validation_x,
        validation_y,
        temperature=1.0,
    )

    nll_after = multiclass_nll(
        validation_x,
        validation_y,
        temperature=fitted_temperature,
    )

    ece_before = multiclass_ece(
        validation_x,
        validation_y,
        temperature=1.0,
    )

    ece_after = multiclass_ece(
        validation_x,
        validation_y,
        temperature=fitted_temperature,
    )

    retained = bool(
        nll_after < nll_before
    )

    used_temperature = (
        fitted_temperature
        if retained
        else 1.0
    )

    temperature_rows.append({
        "Model": model_name,
        "Fitted Temperature": fitted_temperature,
        "Used Temperature": used_temperature,
        "Development NLL Before": nll_before,
        "Development NLL After": nll_after,
        "Development ECE Before": ece_before,
        "Development ECE After": ece_after,
        "Retained": retained,
    })


temperature_summary = pd.DataFrame(
    temperature_rows
)

BERT_TEMPERATURE_WNUT = float(
    temperature_summary.loc[
        temperature_summary["Model"].eq("BERT"),
        "Used Temperature",
    ].iloc[0]
)

SPANNER_TEMPERATURE_WNUT = float(
    temperature_summary.loc[
        temperature_summary["Model"].eq("SpanNER"),
        "Used Temperature",
    ].iloc[0]
)

display(
    temperature_summary.round(6)
)

,Model,Fitted Temperature,Used Temperature,Development NLL Before,Development NLL After,Development ECE Before,Development ECE After,Retained
0,BERT,1.247355,1.247355,0.226688,0.211579,0.022876,0.010052,True
1,SpanNER,0.998191,1.000000,0.041741,0.041751,0.002105,0.002075,False


## 7. Typed Candidates and Boundary Uncertainty

Conformal prediction is defined over a common typed-candidate universe consisting of every contiguous span of one to four original words crossed with each of the six native WNUT entity types.

The two architectures retain their native confidence constructions:

* **BERT:** geometric mean of the temperature-scaled `B-type` probability at the candidate start and the corresponding `I-type` probabilities across subsequent words;
* **SpanNER:** direct probability assigned to the candidate's entity type.

Boundary competition is represented separately for the start and end of each candidate. Candidate-specific rank penalties compare confidence against valid same-type alternatives that modify only one boundary.

For BERT, start uncertainty compares `B-type` and `I-type` evidence at the proposed start, while end uncertainty uses continuation evidence immediately after the proposed end. For SpanNER, start and end uncertainty compare the current candidate with the strongest same-type candidate that changes only the corresponding boundary.

All confidence and uncertainty quantities are constructed from the retained temperature-scaled probability distributions.


### Construct the common typed candidate universe

In [16]:
BOUNDARY_EPSILON = 1e-12


def add_boundary_ranks(candidate_df):
    """Add normalised start- and end-boundary confidence ranks."""
    result = candidate_df.copy()

    # Start alternatives keep sentence, end and entity type fixed.
    start_group = [
        "sentence_id",
        "end",
        "entity_type",
    ]

    # End alternatives keep sentence, start and entity type fixed.
    end_group = [
        "sentence_id",
        "start",
        "entity_type",
    ]

    result["start_group_size"] = (
        result.groupby(
            start_group,
            sort=False,
        )["base_confidence"]
        .transform("size")
    )

    result["end_group_size"] = (
        result.groupby(
            end_group,
            sort=False,
        )["base_confidence"]
        .transform("size")
    )

    result["start_rank"] = (
        result.groupby(
            start_group,
            sort=False,
        )["base_confidence"]
        .rank(
            method="min",
            ascending=False,
        )
    )

    result["end_rank"] = (
        result.groupby(
            end_group,
            sort=False,
        )["base_confidence"]
        .rank(
            method="min",
            ascending=False,
        )
    )

    result["start_rank_penalty"] = np.where(
        result["start_group_size"] > 1,
        (
            result["start_rank"] - 1.0
        )
        / (
            result["start_group_size"] - 1.0
        ),
        0.0,
    )

    result["end_rank_penalty"] = np.where(
        result["end_group_size"] > 1,
        (
            result["end_rank"] - 1.0
        )
        / (
            result["end_group_size"] - 1.0
        ),
        0.0,
    )

    return result


def build_bert_typed_candidates(
    sentence_outputs,
    dataset_split,
):
    """Construct BERT width≤4 typed candidates from scaled BIO probabilities."""
    rows = []

    for sentence_output, example in zip(
        sentence_outputs,
        dataset_split,
    ):
        sentence_id = int(
            sentence_output["sentence_id"]
        )

        tokens = list(
            sentence_output["tokens"]
        )

        # The same scaled probability matrix supplies confidence
        # and both BERT boundary-uncertainty signals.
        probabilities = softmax_numpy(
            sentence_output["word_logits"]
            / BERT_TEMPERATURE_WNUT
        )

        gold_set = {
            (
                int(span["start"]),
                int(span["end"]),
                str(span["type"]),
            )
            for span in example_to_gold_spans(
                example
            )
            if (
                int(span["end"])
                - int(span["start"])
                + 1
                <= MAX_SPAN_WIDTH
            )
        }

        for start in range(len(tokens)):
            maximum_end = min(
                len(tokens) - 1,
                start + MAX_SPAN_WIDTH - 1,
            )

            for end in range(
                start,
                maximum_end + 1,
            ):
                width = end - start + 1

                for entity_type in entity_types:
                    begin_id = label2id[
                        f"B-{entity_type}"
                    ]

                    inside_id = label2id[
                        f"I-{entity_type}"
                    ]

                    component_probabilities = [
                        probabilities[
                            start,
                            begin_id,
                        ]
                    ]

                    if end > start:
                        component_probabilities.extend(
                            probabilities[
                                start + 1:end + 1,
                                inside_id,
                            ].tolist()
                        )

                    # Geometric span confidence used by the
                    # BERT conformal candidate construction.
                    log_confidence = np.mean(
                        np.log(
                            np.clip(
                                component_probabilities,
                                1e-300,
                                1.0,
                            )
                        )
                    )

                    base_confidence = float(
                        np.exp(log_confidence)
                    )

                    begin_probability = float(
                        probabilities[
                            start,
                            begin_id,
                        ]
                    )

                    inside_at_start = float(
                        probabilities[
                            start,
                            inside_id,
                        ]
                    )

                    start_uncertainty = (
                        inside_at_start
                        / (
                            begin_probability
                            + inside_at_start
                            + BOUNDARY_EPSILON
                        )
                    )

                    if end + 1 < len(tokens):
                        end_uncertainty = float(
                            probabilities[
                                end + 1,
                                inside_id,
                            ]
                        )
                    else:
                        end_uncertainty = 0.0

                    identity = (
                        start,
                        end,
                        entity_type,
                    )

                    rows.append({
                        "sentence_id": sentence_id,
                        "start": start,
                        "end": end,
                        "word_length": width,
                        "entity_type": entity_type,
                        "base_confidence": base_confidence,
                        "start_uncertainty": float(
                            start_uncertainty
                        ),
                        "end_uncertainty": float(
                            end_uncertainty
                        ),
                        "is_gold_candidate": (
                            identity in gold_set
                        ),
                    })

    return add_boundary_ranks(
        pd.DataFrame(rows)
    )


def build_spanner_typed_candidates(
    inference_output,
    dataset_split,
):
    """Construct SpanNER typed candidates from the retained temperature."""
    source = inference_output[
        "candidate_logits"
    ].copy()

    logit_columns = [
        f"logit_{spanner_id2label[index]}"
        for index in range(
            len(SPANNER_LABEL_NAMES)
        )
    ]

    logits = source[
        logit_columns
    ].to_numpy(
        dtype=np.float64
    )

    probabilities = softmax_numpy(
        logits / SPANNER_TEMPERATURE_WNUT
    )

    rows = []

    gold_sets = {
        sentence_id: {
            (
                int(span["start"]),
                int(span["end"]),
                str(span["type"]),
            )
            for span in example_to_gold_spans(
                example
            )
            if (
                int(span["end"])
                - int(span["start"])
                + 1
                <= MAX_SPAN_WIDTH
            )
        }
        for sentence_id, example
        in enumerate(dataset_split)
    }

    for row_position, candidate in source.iterrows():
        sentence_id = int(
            candidate["sentence_id"]
        )

        start = int(
            candidate["start"]
        )

        end = int(
            candidate["end"]
        )

        width = int(
            candidate["width"]
        )

        for entity_type in entity_types:
            type_id = spanner_label2id[
                entity_type
            ]

            identity = (
                start,
                end,
                entity_type,
            )

            rows.append({
                "sentence_id": sentence_id,
                "candidate_index": int(
                    candidate["candidate_index"]
                ),
                "start": start,
                "end": end,
                "word_length": width,
                "entity_type": entity_type,
                "base_confidence": float(
                    probabilities[
                        row_position,
                        type_id,
                    ]
                ),
                "is_gold_candidate": (
                    identity
                    in gold_sets[sentence_id]
                ),
            })

    return add_boundary_ranks(
        pd.DataFrame(rows)
    )


# Development candidates are used for M1–M4 method development.
bert_validation_candidates = (
    build_bert_typed_candidates(
        bert_validation_logits,
        wnut_validation,
    )
)

spanner_validation_candidates = (
    build_spanner_typed_candidates(
        spanner_validation_inference,
        wnut_validation,
    )
)

# Conformal-calibration candidates are held for q-hat estimation.
bert_conformal_candidates = (
    build_bert_typed_candidates(
        bert_conformal_logits,
        wnut_conformal_calibration,
    )
)

spanner_conformal_candidates = (
    build_spanner_typed_candidates(
        spanner_conformal_inference,
        wnut_conformal_calibration,
    )
)

print("Matched typed-candidate universes constructed.")

Matched typed-candidate universes constructed.


### SpanNER boundary competition

In [17]:
def strongest_excluding_self(
    values,
    group_keys,
):
    """Return the strongest other value within each boundary group."""
    values = pd.Series(
        values.to_numpy(dtype=float),
        index=values.index,
    )

    grouping = [
        group_keys[column]
        for column in group_keys.columns
    ]

    group_max = values.groupby(
        grouping
    ).transform("max")

    group_size = values.groupby(
        grouping
    ).transform("size")

    is_group_max = np.isclose(
        values.to_numpy(dtype=float),
        group_max.to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-15,
    )

    max_count = (
        pd.Series(
            is_group_max.astype(int),
            index=values.index,
        )
        .groupby(grouping)
        .transform("sum")
    )

    below_max = values.where(
        values < group_max,
        other=-np.inf,
    )

    second_distinct = (
        below_max
        .groupby(grouping)
        .transform("max")
    )

    competitor = group_max.copy()

    # If the current row is the unique maximum, use the
    # strongest remaining candidate rather than itself.
    unique_max = (
        is_group_max
        & (
            max_count.to_numpy(
                dtype=int
            )
            == 1
        )
    )

    competitor.loc[
        unique_max
    ] = second_distinct.loc[
        unique_max
    ]

    competitor.loc[
        group_size <= 1
    ] = np.nan

    competitor.replace(
        -np.inf,
        np.nan,
        inplace=True,
    )

    return competitor


def add_spanner_boundary_uncertainty(
    candidate_df,
):
    """Attach candidate-local start and end competition uncertainty."""
    result = candidate_df.copy()

    # Competing starts: same sentence, end and type.
    start_keys = result[
        [
            "sentence_id",
            "end",
            "entity_type",
        ]
    ]

    # Competing ends: same sentence, start and type.
    end_keys = result[
        [
            "sentence_id",
            "start",
            "entity_type",
        ]
    ]

    result[
        "start_competitor_probability"
    ] = strongest_excluding_self(
        result["base_confidence"],
        start_keys,
    )

    result[
        "end_competitor_probability"
    ] = strongest_excluding_self(
        result["base_confidence"],
        end_keys,
    )

    current = result[
        "base_confidence"
    ].to_numpy(dtype=float)

    start_competitor = result[
        "start_competitor_probability"
    ].to_numpy(dtype=float)

    end_competitor = result[
        "end_competitor_probability"
    ].to_numpy(dtype=float)

    result["start_uncertainty"] = np.where(
        np.isfinite(start_competitor),
        start_competitor
        / (
            current
            + start_competitor
            + BOUNDARY_EPSILON
        ),
        0.0,
    )

    result["end_uncertainty"] = np.where(
        np.isfinite(end_competitor),
        end_competitor
        / (
            current
            + end_competitor
            + BOUNDARY_EPSILON
        ),
        0.0,
    )

    return result


spanner_validation_candidates = (
    add_spanner_boundary_uncertainty(
        spanner_validation_candidates
    )
)

spanner_conformal_candidates = (
    add_spanner_boundary_uncertainty(
        spanner_conformal_candidates
    )
)


candidate_summary = pd.DataFrame([
    {
        "Model": "BERT",
        "Population": "Development",
        "Typed candidates": len(
            bert_validation_candidates
        ),
        "Gold candidates": int(
            bert_validation_candidates[
                "is_gold_candidate"
            ].sum()
        ),
    },
    {
        "Model": "SpanNER",
        "Population": "Development",
        "Typed candidates": len(
            spanner_validation_candidates
        ),
        "Gold candidates": int(
            spanner_validation_candidates[
                "is_gold_candidate"
            ].sum()
        ),
    },
    {
        "Model": "BERT",
        "Population": "Conformal calibration",
        "Typed candidates": len(
            bert_conformal_candidates
        ),
        "Gold candidates": int(
            bert_conformal_candidates[
                "is_gold_candidate"
            ].sum()
        ),
    },
    {
        "Model": "SpanNER",
        "Population": "Conformal calibration",
        "Typed candidates": len(
            spanner_conformal_candidates
        ),
        "Gold candidates": int(
            spanner_conformal_candidates[
                "is_gold_candidate"
            ].sum()
        ),
    },
])

display(candidate_summary)

,Model,Population,Typed candidates,Gold candidates
0,BERT,Development,174738,418
1,SpanNER,Development,174738,418
2,BERT,Conformal calibration,66078,164
3,SpanNER,Conformal calibration,66078,164


## 8. Validation-Only M1–M4 Development and Method Freeze

M1–M4 are developed exclusively on the 505-sentence WNUT development population.

Five duplicate-safe folds are used for cross-fitted pseudo-conformal evaluation. For each held-out fold, the remaining four folds provide a temporary calibration population. The sentence-level calibration score is the maximum nonconformity among all representable gold entities in that sentence; sentences containing no representable gold entity receive score zero.

The candidate-level scores are:

### M1- Confidence-Only Baseline

$$
M1 = a_{\text{base}}
$$

M1 uses candidate confidence only and contains no explicit boundary regularisation.

### M2- Shared Fixed Boundary Regularisation

$$
M2 =
a_{\text{base}}
+
\lambda_{\text{shared}}
\frac{R_s + R_e}{2}
$$

The same regularisation strength is applied to both boundaries.

### M3- Separate Fixed Start/End Regularisation

$$
M3 =
a_{\text{base}}
+
\frac{
\lambda_s R_s
+
\lambda_e R_e
}{2}
$$

M3 allows the start and end boundaries to receive different fixed regularisation strengths.

### M4- Uncertainty-Adaptive Boundary Regularisation

$$
M4 =
a_{\text{base}}
+
\frac{
\lambda_s(1-u_s)R_s
+
\lambda_e(1-u_e)R_e
}{2}
$$

The regularisation search spaces are inherited from the completed CoNLL development rather than being designed after observing WNUT results. BERT uses the original compact grid, while SpanNER uses the final expanded CoNLL grid.

The primary development requirement is 90% cross-fitted representable-gold coverage. If no configuration reaches 90%, the predeclared 89% fallback is permitted. If neither level is reached, selection is restricted to configurations attaining the maximum available coverage.

After this section, the selected M1–M4 parameters are frozen. The independent 201-sentence conformal-calibration population is used only afterwards to estimate the final method-specific conformal thresholds.

In [19]:
METHODS = ["M1", "M2", "M3", "M4"]

TARGET_COVERAGES = [0.90, 0.95, 0.99]
PRIMARY_TUNING_COVERAGE = 0.90
COVERAGE_FALLBACK_TOLERANCE = 0.01
N_TUNING_FOLDS = 5

# These are the fixed lambda grids inherited from the CoNLL development.
BERT_LAMBDA_VALUES = [
    0.0,
    0.01,
    0.025,
    0.05,
    0.10,
    0.20,
]

SPANNER_LAMBDA_VALUES = [
    0.0,
    0.01,
    0.025,
    0.05,
    0.10,
    0.20,
    0.30,
    0.40,
    0.60,
    0.80,
    1.00,
]


def make_duplicate_safe_folds(
    dataset_split,
    n_folds,
    seed,
):
    """Assign exact duplicate token sequences to the same tuning fold."""
    hash_groups = defaultdict(list)

    for sentence_id, example in enumerate(dataset_split):
        # sentence_hash() was defined during the WNUT split construction.
        hash_value = sentence_hash(
            example["tokens"]
        )

        hash_groups[hash_value].append(
            sentence_id
        )

    groups = list(
        hash_groups.values()
    )

    local_rng = random.Random(seed)
    local_rng.shuffle(groups)

    fold_members = [
        []
        for _ in range(n_folds)
    ]

    # Allocate each complete duplicate group to the currently
    # smallest fold so identical sentences cannot cross folds.
    for group in groups:
        target_fold = min(
            range(n_folds),
            key=lambda fold_id: len(
                fold_members[fold_id]
            ),
        )

        fold_members[
            target_fold
        ].extend(group)

    rows = []

    for fold_id, members in enumerate(
        fold_members
    ):
        for sentence_id in members:
            rows.append({
                "sentence_id": int(
                    sentence_id
                ),
                "fold_id": int(
                    fold_id
                ),
            })

    result = (
        pd.DataFrame(rows)
        .sort_values("sentence_id")
        .reset_index(drop=True)
    )

    if len(result) != len(dataset_split):
        raise RuntimeError(
            "Development fold assignment is incomplete."
        )

    return result


validation_fold_df = make_duplicate_safe_folds(
    wnut_validation,
    N_TUNING_FOLDS,
    SEED,
)

fold_summary_df = (
    validation_fold_df
    .groupby(
        "fold_id",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "Sentences"
        }
    )
)

display(
    fold_summary_df
)


def make_configuration_grid(
    lambda_values,
):
    """Create the fixed M1-M4 regularisation search grid."""
    positive_values = [
        value
        for value in lambda_values
        if value > 0.0
    ]

    records = [
        {
            "method": "M1",
            "config_id": "M1_raw",
            "lambda_shared": 0.0,
            "lambda_start": 0.0,
            "lambda_end": 0.0,
        }
    ]

    # M2 uses one shared start/end regularisation coefficient.
    for lambda_shared in positive_values:
        records.append({
            "method": "M2",
            "config_id": (
                f"M2_shared_{lambda_shared:g}"
            ),
            "lambda_shared": float(
                lambda_shared
            ),
            "lambda_start": 0.0,
            "lambda_end": 0.0,
        })

    # M3 and M4 allow separate start/end coefficients.
    # The all-zero pair is excluded because it is identical to M1.
    for method in ["M3", "M4"]:
        for lambda_start in lambda_values:
            for lambda_end in lambda_values:
                if (
                    lambda_start == 0.0
                    and lambda_end == 0.0
                ):
                    continue

                records.append({
                    "method": method,
                    "config_id": (
                        f"{method}_start_"
                        f"{lambda_start:g}"
                        f"_end_{lambda_end:g}"
                    ),
                    "lambda_shared": 0.0,
                    "lambda_start": float(
                        lambda_start
                    ),
                    "lambda_end": float(
                        lambda_end
                    ),
                })

    return pd.DataFrame(
        records
    )


bert_configuration_grid = (
    make_configuration_grid(
        BERT_LAMBDA_VALUES
    )
)

spanner_configuration_grid = (
    make_configuration_grid(
        SPANNER_LAMBDA_VALUES
    )
)

grid_summary_df = pd.concat(
    [
        (
            bert_configuration_grid
            .groupby(
                "method",
                as_index=False,
            )
            .size()
            .assign(Model="BERT")
        ),
        (
            spanner_configuration_grid
            .groupby(
                "method",
                as_index=False,
            )
            .size()
            .assign(Model="SpanNER")
        ),
    ],
    ignore_index=True,
).rename(
    columns={
        "size": "Configurations"
    }
)

grid_summary_df = grid_summary_df[
    [
        "Model",
        "method",
        "Configurations",
    ]
].rename(
    columns={
        "method": "Method"
    }
)

display(
    grid_summary_df
)

,fold_id,Sentences
0,0,101
1,1,101
2,2,101
3,3,101
4,4,101


,Model,Method,Configurations
0,BERT,M1,1
1,BERT,M2,5
2,BERT,M3,35
3,BERT,M4,35
4,SpanNER,M1,1
5,SpanNER,M2,10
6,SpanNER,M3,120
7,SpanNER,M4,120


### Cross-fitted M1–M4 evaluation

In [20]:
# Confidence-only nonconformity shared by M1-M4.
bert_validation_candidates[
    "base_nonconformity"
] = (
    1.0
    - bert_validation_candidates[
        "base_confidence"
    ]
)

spanner_validation_candidates[
    "base_nonconformity"
] = (
    1.0
    - spanner_validation_candidates[
        "base_confidence"
    ]
)


def full_gold_counts_for_split(dataset_split):
    """Count all native gold entities in each sentence."""
    return np.asarray(
        [
            len(
                example_to_gold_spans(
                    example
                )
            )
            for example in dataset_split
        ],
        dtype=np.int64,
    )


development_full_gold_counts = (
    full_gold_counts_for_split(
        wnut_validation
    )
)


def build_tuning_context(
    candidate_df,
    model_name,
):
    """Prepare compact arrays used repeatedly during lambda search."""
    sentence_count = len(
        wnut_validation
    )

    fold_lookup = (
        validation_fold_df
        .set_index("sentence_id")[
            "fold_id"
        ]
    )

    sentence_folds = np.asarray(
        [
            int(
                fold_lookup.loc[
                    sentence_id
                ]
            )
            for sentence_id
            in range(sentence_count)
        ],
        dtype=np.int64,
    )

    candidate_sentence_ids = (
        candidate_df[
            "sentence_id"
        ].to_numpy(
            dtype=np.int64
        )
    )

    candidate_folds = (
        sentence_folds[
            candidate_sentence_ids
        ]
    )

    gold_mask = (
        candidate_df[
            "is_gold_candidate"
        ].to_numpy(
            dtype=bool
        )
    )

    gold_sentence_ids = (
        candidate_sentence_ids[
            gold_mask
        ]
    )

    representable_gold_counts = (
        np.bincount(
            gold_sentence_ids,
            minlength=sentence_count,
        )
    )

    # A sentence is fully structurally representable only when
    # all of its native gold entities fall inside width≤4 support.
    fully_representable = (
        representable_gold_counts
        == development_full_gold_counts
    )

    start_rank = (
        candidate_df[
            "start_rank_penalty"
        ].to_numpy(
            dtype=np.float64
        )
    )

    end_rank = (
        candidate_df[
            "end_rank_penalty"
        ].to_numpy(
            dtype=np.float64
        )
    )

    start_uncertainty = (
        candidate_df[
            "start_uncertainty"
        ].to_numpy(
            dtype=np.float64
        )
    )

    end_uncertainty = (
        candidate_df[
            "end_uncertainty"
        ].to_numpy(
            dtype=np.float64
        )
    )

    return {
        "model_name": model_name,
        "sentence_count": sentence_count,
        "sentence_folds": sentence_folds,
        "candidate_sentence_ids": (
            candidate_sentence_ids
        ),
        "candidate_folds": (
            candidate_folds
        ),
        "gold_mask": gold_mask,
        "gold_sentence_ids": (
            gold_sentence_ids
        ),
        "representable_gold_counts": (
            representable_gold_counts
        ),
        "fully_representable": (
            fully_representable
        ),
        "base": (
            candidate_df[
                "base_nonconformity"
            ].to_numpy(
                dtype=np.float64
            )
        ),

        # M2: shared fixed rank component.
        "shared_rank_component": (
            start_rank
            + end_rank
        ) / 2.0,

        # M3: separate fixed start/end components.
        "fixed_start_component": (
            start_rank / 2.0
        ),
        "fixed_end_component": (
            end_rank / 2.0
        ),

        # M4: uncertainty-adaptive start/end components.
        "adaptive_start_component": (
            (
                1.0
                - start_uncertainty
            )
            * start_rank
            / 2.0
        ),
        "adaptive_end_component": (
            (
                1.0
                - end_uncertainty
            )
            * end_rank
            / 2.0
        ),
    }


bert_tuning_context = (
    build_tuning_context(
        bert_validation_candidates,
        "BERT",
    )
)

spanner_tuning_context = (
    build_tuning_context(
        spanner_validation_candidates,
        "SpanNER",
    )
)


def finite_sample_q(
    scores,
    target_coverage,
):
    """Return the finite-sample split-conformal order statistic."""
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    n = len(scores)

    if n == 0:
        raise ValueError(
            "Calibration scores cannot be empty."
        )

    k = int(
        np.ceil(
            (n + 1)
            * float(target_coverage)
        )
    )

    k = min(
        max(k, 1),
        n,
    )

    return float(
        np.partition(
            scores,
            k - 1,
        )[k - 1]
    )


def construct_candidate_score(
    context,
    method,
    lambda_shared=0.0,
    lambda_start=0.0,
    lambda_end=0.0,
):
    """Construct one candidate-level M1-M4 nonconformity score."""
    base = context["base"]

    if method == "M1":
        return base.copy()

    if method == "M2":
        return (
            base
            + float(lambda_shared)
            * context[
                "shared_rank_component"
            ]
        )

    if method == "M3":
        return (
            base
            + float(lambda_start)
            * context[
                "fixed_start_component"
            ]
            + float(lambda_end)
            * context[
                "fixed_end_component"
            ]
        )

    if method == "M4":
        return (
            base
            + float(lambda_start)
            * context[
                "adaptive_start_component"
            ]
            + float(lambda_end)
            * context[
                "adaptive_end_component"
            ]
        )

    raise ValueError(
        f"Unknown conformal method: {method}"
    )


def evaluate_configuration_cross_fitted(
    context,
    method,
    lambda_shared=0.0,
    lambda_start=0.0,
    lambda_end=0.0,
):
    """
    Evaluate one fixed score configuration using five-fold
    development-only pseudo-conformal calibration.
    """
    candidate_scores = (
        construct_candidate_score(
            context=context,
            method=method,
            lambda_shared=lambda_shared,
            lambda_start=lambda_start,
            lambda_end=lambda_end,
        )
    )

    sentence_count = (
        context[
            "sentence_count"
        ]
    )

    # Sentence-level calibration score:
    # maximum score among representable gold entities.
    # Sentences with no representable gold entity remain at zero.
    sentence_gold_scores = np.zeros(
        sentence_count,
        dtype=np.float64,
    )

    np.maximum.at(
        sentence_gold_scores,
        context[
            "gold_sentence_ids"
        ],
        candidate_scores[
            context[
                "gold_mask"
            ]
        ],
    )

    covered = np.zeros(
        sentence_count,
        dtype=bool,
    )

    set_sizes = np.zeros(
        sentence_count,
        dtype=np.int64,
    )

    included_gold_counts = np.zeros(
        sentence_count,
        dtype=np.int64,
    )

    for fold_id in range(
        N_TUNING_FOLDS
    ):
        # The other four folds act as temporary calibration data.
        pseudo_calibration_mask = (
            context[
                "sentence_folds"
            ]
            != fold_id
        )

        heldout_sentence_mask = (
            context[
                "sentence_folds"
            ]
            == fold_id
        )

        q_hat = finite_sample_q(
            sentence_gold_scores[
                pseudo_calibration_mask
            ],
            PRIMARY_TUNING_COVERAGE,
        )

        heldout_sentence_ids = (
            np.flatnonzero(
                heldout_sentence_mask
            )
        )

        covered[
            heldout_sentence_ids
        ] = (
            sentence_gold_scores[
                heldout_sentence_ids
            ]
            <= q_hat
        )

        # Candidate prediction-set sizes in the held-out fold.
        candidate_fold_mask = (
            context[
                "candidate_folds"
            ]
            == fold_id
        )

        fold_sentence_ids = (
            context[
                "candidate_sentence_ids"
            ][candidate_fold_mask]
        )

        fold_scores = (
            candidate_scores[
                candidate_fold_mask
            ]
        )

        included_candidate_mask = (
            fold_scores
            <= q_hat
        )

        candidate_counts = np.bincount(
            fold_sentence_ids[
                included_candidate_mask
            ],
            minlength=sentence_count,
        )

        set_sizes[
            heldout_sentence_ids
        ] = candidate_counts[
            heldout_sentence_ids
        ]

        # Number of representable gold candidates retained.
        gold_fold_mask = (
            candidate_fold_mask
            & context[
                "gold_mask"
            ]
        )

        gold_sentence_ids = (
            context[
                "candidate_sentence_ids"
            ][gold_fold_mask]
        )

        gold_scores = (
            candidate_scores[
                gold_fold_mask
            ]
        )

        gold_counts = np.bincount(
            gold_sentence_ids[
                gold_scores <= q_hat
            ],
            minlength=sentence_count,
        )

        included_gold_counts[
            heldout_sentence_ids
        ] = gold_counts[
            heldout_sentence_ids
        ]

    # Full-gold coverage additionally requires that no native
    # gold entity lies outside the width≤4 candidate universe.
    full_gold_covered = (
        covered
        & context[
            "fully_representable"
        ]
    )

    return {
        "representable_gold_coverage": float(
            covered.mean()
        ),
        "full_gold_coverage": float(
            full_gold_covered.mean()
        ),
        "mean_set_size": float(
            set_sizes.mean()
        ),
        "median_set_size": float(
            np.median(
                set_sizes
            )
        ),
        "mean_extra_candidates": float(
            np.mean(
                set_sizes
                - included_gold_counts
            )
        ),
    }

### Select and freeze M1–M4

In [21]:
def run_configuration_grid(
    context,
    configuration_grid,
):
    """Evaluate every predeclared M1-M4 configuration."""
    rows = []

    for configuration in tqdm(
        configuration_grid.itertuples(index=False),
        total=len(configuration_grid),
        desc=f"{context['model_name']} M1-M4 validation grid",
    ):
        metrics = evaluate_configuration_cross_fitted(
            context=context,
            method=configuration.method,
            lambda_shared=configuration.lambda_shared,
            lambda_start=configuration.lambda_start,
            lambda_end=configuration.lambda_end,
        )

        rows.append({
            "model": context["model_name"],
            "method": configuration.method,
            "config_id": configuration.config_id,
            "lambda_shared": float(configuration.lambda_shared),
            "lambda_start": float(configuration.lambda_start),
            "lambda_end": float(configuration.lambda_end),
            **metrics,
        })

    return pd.DataFrame(rows)


# Development-only search over the frozen CoNLL-derived grids.
bert_tuning_results = run_configuration_grid(
    bert_tuning_context,
    bert_configuration_grid,
)

spanner_tuning_results = run_configuration_grid(
    spanner_tuning_context,
    spanner_configuration_grid,
)

print("Validation-only M1-M4 development complete.")


def select_configuration(method_results):
    """Apply the frozen WNUT coverage-efficiency selection rule."""
    target = PRIMARY_TUNING_COVERAGE
    fallback = target - COVERAGE_FALLBACK_TOLERANCE

    # Primary requirement: at least 90% representable-gold coverage.
    primary_pool = method_results.loc[
        method_results["representable_gold_coverage"] >= target
    ].copy()

    if len(primary_pool) > 0:
        pool = primary_pool
        status = "meets_90_percent"
        coverage_floor = target

    else:
        # Predeclared fallback: within one percentage point of target.
        fallback_pool = method_results.loc[
            method_results["representable_gold_coverage"] >= fallback
        ].copy()

        if len(fallback_pool) > 0:
            pool = fallback_pool
            status = "fallback_within_1_percentage_point"
            coverage_floor = fallback

        else:
            # If neither floor is attainable, compare only configurations
            # reaching the maximum observed coverage for that method.
            maximum_coverage = float(
                method_results[
                    "representable_gold_coverage"
                ].max()
            )

            pool = method_results.loc[
                np.isclose(
                    method_results[
                        "representable_gold_coverage"
                    ],
                    maximum_coverage,
                )
            ].copy()

            status = "maximum_available_coverage"
            coverage_floor = maximum_coverage

    pool["regularisation_sum"] = (
        pool["lambda_shared"]
        + pool["lambda_start"]
        + pool["lambda_end"]
    )

    # Prefer the smallest prediction set, then higher coverage,
    # weaker regularisation, and finally deterministic config ID.
    selected = (
        pool.sort_values(
            [
                "mean_set_size",
                "representable_gold_coverage",
                "regularisation_sum",
                "config_id",
            ],
            ascending=[
                True,
                False,
                True,
                True,
            ],
        )
        .iloc[0]
    )

    return selected, status, coverage_floor


selected_rows = []

for model_name, tuning_results in [
    ("BERT", bert_tuning_results),
    ("SpanNER", spanner_tuning_results),
]:
    for method in METHODS:
        method_results = tuning_results.loc[
            tuning_results["method"].eq(method)
        ].copy()

        (
            selected,
            status,
            coverage_floor,
        ) = select_configuration(
            method_results
        )

        selected_rows.append({
            "Model": model_name,
            "Method": method,
            "Selection Status": status,
            "Coverage Floor": float(coverage_floor),
            "Lambda Shared": float(
                selected["lambda_shared"]
            ),
            "Lambda Start": float(
                selected["lambda_start"]
            ),
            "Lambda End": float(
                selected["lambda_end"]
            ),
            "Representable-Gold Coverage": float(
                selected[
                    "representable_gold_coverage"
                ]
            ),
            "Full-Gold Coverage": float(
                selected[
                    "full_gold_coverage"
                ]
            ),
            "Mean Set Size": float(
                selected["mean_set_size"]
            ),
            "Median Set Size": float(
                selected["median_set_size"]
            ),
        })


selected_method_df = pd.DataFrame(
    selected_rows
)

display(
    selected_method_df.round(6)
)


# Compact parameter dictionary reused during independent
# conformal calibration and final WNUT test evaluation.
method_parameters = {}

for _, row in selected_method_df.iterrows():
    model_name = row["Model"]
    method = row["Method"]

    method_parameters.setdefault(
        model_name,
        {}
    )

    method_parameters[
        model_name
    ][method] = {
        "lambda_shared": float(
            row["Lambda Shared"]
        ),
        "lambda_start": float(
            row["Lambda Start"]
        ),
        "lambda_end": float(
            row["Lambda End"]
        ),
    }


print("\nM1-M4 method specification frozen.")

BERT M1-M4 validation grid:   0%|          | 0/76 [00:00<?, ?it/s]

SpanNER M1-M4 validation grid:   0%|          | 0/251 [00:00<?, ?it/s]

Validation-only M1-M4 development complete.


,Model,Method,Selection Status,Coverage Floor,Lambda Shared,Lambda Start,Lambda End,Representable-Gold Coverage,Full-Gold Coverage,Mean Set Size,Median Set Size
0,BERT,M1,meets_90_percent,0.90,0.000,0.0,0.000,0.90495,0.897030,17.382178,8.0
1,BERT,M2,meets_90_percent,0.90,0.025,0.0,0.000,0.90099,0.893069,15.108911,10.0
2,BERT,M3,meets_90_percent,0.90,0.000,0.1,0.010,0.90099,0.895050,14.469307,11.0
3,BERT,M4,meets_90_percent,0.90,0.000,0.1,0.000,0.90297,0.895050,15.003960,9.0
4,SpanNER,M1,fallback_within_1_percentage_point,0.89,0.000,0.0,0.000,0.89703,0.891089,12.247525,10.0
5,SpanNER,M2,meets_90_percent,0.90,0.025,0.0,0.000,0.90297,0.897030,11.083168,9.0
6,SpanNER,M3,meets_90_percent,0.90,0.000,0.1,0.025,0.90297,0.895050,10.623762,9.0
7,SpanNER,M4,meets_90_percent,0.90,0.000,0.2,0.050,0.90099,0.895050,11.314851,9.0



M1-M4 method specification frozen.


## 9. Independent Conformal Calibration

After all M1–M4 regularisation parameters have been frozen, the independent 201-sentence conformal-calibration population is used to estimate the final conformal thresholds.

For each architecture and method, one sentence-level calibration score is formed by taking the maximum nonconformity score among all representable gold typed entities in that sentence. If a sentence contains no representable gold entity, its calibration score is zero.

Separate finite-sample conformal thresholds are estimated for target coverages of 90%, 95% and 99%. Each architecture–method combination receives its own threshold. No model parameter, temperature, uncertainty definition or λ value is modified after this stage.

### Score the 201 calibration sentences and estimate q̂

In [22]:
def add_frozen_method_scores(
    candidate_df,
    model_name,
):
    """Apply the frozen M1-M4 parameters to one candidate population."""
    result = candidate_df.copy()

    result["base_nonconformity"] = (
        1.0
        - result["base_confidence"]
    )

    for method in METHODS:
        parameters = (
            method_parameters[
                model_name
            ][method]
        )

        base = result[
            "base_nonconformity"
        ].to_numpy(
            dtype=np.float64
        )

        start_rank = result[
            "start_rank_penalty"
        ].to_numpy(
            dtype=np.float64
        )

        end_rank = result[
            "end_rank_penalty"
        ].to_numpy(
            dtype=np.float64
        )

        if method == "M1":
            score = base

        elif method == "M2":
            lambda_shared = float(
                parameters[
                    "lambda_shared"
                ]
            )

            score = (
                base
                + lambda_shared
                * (
                    start_rank
                    + end_rank
                )
                / 2.0
            )

        elif method == "M3":
            lambda_start = float(
                parameters[
                    "lambda_start"
                ]
            )

            lambda_end = float(
                parameters[
                    "lambda_end"
                ]
            )

            score = (
                base
                + (
                    lambda_start
                    * start_rank
                    + lambda_end
                    * end_rank
                )
                / 2.0
            )

        elif method == "M4":
            lambda_start = float(
                parameters[
                    "lambda_start"
                ]
            )

            lambda_end = float(
                parameters[
                    "lambda_end"
                ]
            )

            start_uncertainty = result[
                "start_uncertainty"
            ].to_numpy(
                dtype=np.float64
            )

            end_uncertainty = result[
                "end_uncertainty"
            ].to_numpy(
                dtype=np.float64
            )

            score = (
                base
                + (
                    lambda_start
                    * (
                        1.0
                        - start_uncertainty
                    )
                    * start_rank
                    + lambda_end
                    * (
                        1.0
                        - end_uncertainty
                    )
                    * end_rank
                )
                / 2.0
            )

        else:
            raise ValueError(
                f"Unknown method: {method}"
            )

        result[
            f"score_{method}"
        ] = score

    return result


# Apply the already-frozen M1-M4 definitions.
bert_conformal_scored = (
    add_frozen_method_scores(
        bert_conformal_candidates,
        "BERT",
    )
)

spanner_conformal_scored = (
    add_frozen_method_scores(
        spanner_conformal_candidates,
        "SpanNER",
    )
)


def calibrate_conformal_thresholds(
    candidate_df,
    model_name,
    sentence_count,
):
    """
    Fit method-specific finite-sample q-hat values from
    sentence-level representable-gold calibration scores.
    """
    sentence_ids = list(
        range(sentence_count)
    )

    rows = []

    for method in METHODS:
        score_column = (
            f"score_{method}"
        )

        # One simultaneous score per sentence:
        # worst representable-gold candidate score.
        sentence_scores = (
            candidate_df.loc[
                candidate_df[
                    "is_gold_candidate"
                ],
                [
                    "sentence_id",
                    score_column,
                ],
            ]
            .groupby(
                "sentence_id"
            )[score_column]
            .max()
            .reindex(
                sentence_ids
            )
            .fillna(0.0)
            .to_numpy(
                dtype=np.float64
            )
        )

        for target in TARGET_COVERAGES:
            q_hat = finite_sample_q(
                sentence_scores,
                target,
            )

            rows.append({
                "Model": model_name,
                "Method": method,
                "Target Coverage": float(
                    target
                ),
                "Calibration Sentences": int(
                    sentence_count
                ),
                "q_hat": float(
                    q_hat
                ),
            })

    return pd.DataFrame(rows)


bert_thresholds = (
    calibrate_conformal_thresholds(
        bert_conformal_scored,
        "BERT",
        len(
            wnut_conformal_calibration
        ),
    )
)

spanner_thresholds = (
    calibrate_conformal_thresholds(
        spanner_conformal_scored,
        "SpanNER",
        len(
            wnut_conformal_calibration
        ),
    )
)

conformal_threshold_df = (
    pd.concat(
        [
            bert_thresholds,
            spanner_thresholds,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "Model",
            "Method",
            "Target Coverage",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    conformal_threshold_df.round(6)
)

# Save the only calibration artefact required by final test evaluation.
conformal_threshold_df.to_csv(
    CONFORMAL_DIR
    / "wnut_conformal_thresholds.csv",
    index=False,
)

print(
    "\nIndependent conformal calibration complete."
)
print(
    "M1-M4 parameters remain frozen."
)

,Model,Method,Target Coverage,Calibration Sentences,q_hat
0,BERT,M1,0.90,201,0.986917
1,BERT,M1,0.95,201,0.995553
2,BERT,M1,0.99,201,0.998472
3,BERT,M2,0.90,201,0.990774
4,BERT,M2,0.95,201,0.998584
5,BERT,M2,0.99,201,1.010972
6,BERT,M3,0.90,201,0.992797
7,BERT,M3,0.95,201,0.998584
8,BERT,M3,0.99,201,1.041956
9,BERT,M4,0.90,201,0.990774



Independent conformal calibration complete.
M1-M4 parameters remain frozen.


## 10. Final WNUT-17 Test Evaluation

All modelling and conformal decisions are now frozen. The official 1,287-sentence WNUT-17 test population is therefore used once for final evaluation, with no subsequent retuning.

Two result families are reported:

1. **NER performance:** strict typed-span precision, recall and F1 under both native support and the shared width≤4 support.
2. **Conformal performance:** empirical representable-gold coverage and mean typed prediction-set size for M1–M4 at nominal 90%, 95% and 99% targets.

The final conformal evaluation uses the frozen architecture-specific temperatures, boundary-uncertainty definitions, M1–M4 λ values and independently calibrated method-specific q̂ thresholds.

In [23]:
if not RUN_FINAL_WNUT_TEST:
    raise RuntimeError(
        "Final WNUT test evaluation is disabled."
    )

# The dataset was loaded earlier, but the official test population
# is selected for experimental use only now, after method freeze.
wnut_test = wnut["test"]

assert len(wnut_test) == 1287

print(
    "Official WNUT test sentences:",
    len(wnut_test),
)


# Frozen inference only: no parameter updates occur below.
bert_test_logits = bert_word_logits_for_split(
    wnut_test,
    "BERT logits: final WNUT test",
)

spanner_test_inference = spanner_logits_for_split(
    wnut_test,
    "SpanNER logits: final WNUT test",
)


# Recover BERT's native BIO point predictions.
bert_test_predicted_sets = []
bert_test_gold_sets = []

for example, output in zip(
    wnut_test,
    bert_test_logits,
):
    predicted_ids = (
        output["word_logits"]
        .argmax(axis=1)
    )

    predicted_tags = [
        id2label[int(label_id)]
        for label_id in predicted_ids
    ]

    gold_tags = [
        id2label[int(label_id)]
        for label_id in example["ner_tags"]
    ]

    bert_test_predicted_sets.append(
        span_set_from_tags(
            predicted_tags
        )
    )

    bert_test_gold_sets.append(
        span_set_from_tags(
            gold_tags
        )
    )


bert_test_native_metrics = (
    metrics_from_span_sets(
        bert_test_predicted_sets,
        bert_test_gold_sets,
    )
)

spanner_test_native_metrics = (
    metrics_from_span_sets(
        spanner_test_inference[
            "decoded_sets"
        ],
        spanner_test_inference[
            "gold_sets"
        ],
    )
)


def restrict_span_sets_to_width4(
    span_sets,
):
    """Restrict typed spans to the common width≤4 support."""
    return [
        {
            span
            for span in sentence_spans
            if (
                span[1]
                - span[0]
                + 1
                <= MAX_SPAN_WIDTH
            )
        }
        for sentence_spans
        in span_sets
    ]


bert_test_width4_metrics = (
    metrics_from_span_sets(
        restrict_span_sets_to_width4(
            bert_test_predicted_sets
        ),
        restrict_span_sets_to_width4(
            bert_test_gold_sets
        ),
    )
)

spanner_test_width4_metrics = (
    metrics_from_span_sets(
        restrict_span_sets_to_width4(
            spanner_test_inference[
                "decoded_sets"
            ]
        ),
        restrict_span_sets_to_width4(
            spanner_test_inference[
                "gold_sets"
            ]
        ),
    )
)


final_wnut_baseline_df = pd.DataFrame([
    {
        "Model": "BERT",
        "Support": "Native",
        "Precision": (
            bert_test_native_metrics[
                "precision"
            ]
        ),
        "Recall": (
            bert_test_native_metrics[
                "recall"
            ]
        ),
        "F1": (
            bert_test_native_metrics[
                "f1"
            ]
        ),
        "TP": bert_test_native_metrics["tp"],
        "FP": bert_test_native_metrics["fp"],
        "FN": bert_test_native_metrics["fn"],
    },
    {
        "Model": "SpanNER",
        "Support": "Native",
        "Precision": (
            spanner_test_native_metrics[
                "precision"
            ]
        ),
        "Recall": (
            spanner_test_native_metrics[
                "recall"
            ]
        ),
        "F1": (
            spanner_test_native_metrics[
                "f1"
            ]
        ),
        "TP": spanner_test_native_metrics["tp"],
        "FP": spanner_test_native_metrics["fp"],
        "FN": spanner_test_native_metrics["fn"],
    },
    {
        "Model": "BERT",
        "Support": "Width ≤4",
        "Precision": (
            bert_test_width4_metrics[
                "precision"
            ]
        ),
        "Recall": (
            bert_test_width4_metrics[
                "recall"
            ]
        ),
        "F1": (
            bert_test_width4_metrics[
                "f1"
            ]
        ),
        "TP": bert_test_width4_metrics["tp"],
        "FP": bert_test_width4_metrics["fp"],
        "FN": bert_test_width4_metrics["fn"],
    },
    {
        "Model": "SpanNER",
        "Support": "Width ≤4",
        "Precision": (
            spanner_test_width4_metrics[
                "precision"
            ]
        ),
        "Recall": (
            spanner_test_width4_metrics[
                "recall"
            ]
        ),
        "F1": (
            spanner_test_width4_metrics[
                "f1"
            ]
        ),
        "TP": spanner_test_width4_metrics["tp"],
        "FP": spanner_test_width4_metrics["fp"],
        "FN": spanner_test_width4_metrics["fn"],
    },
])

print(
    "\nFinal WNUT point-prediction results:"
)

display(
    final_wnut_baseline_df.round(6)
)

Official WNUT test sentences: 1287


BERT logits: final WNUT test:   0%|          | 0/1287 [00:00<?, ?it/s]

SpanNER logits: final WNUT test:   0%|          | 0/1287 [00:00<?, ?it/s]


Final WNUT point-prediction results:


,Model,Support,Precision,Recall,F1,TP,FP,FN
0,BERT,Native,0.535831,0.304912,0.388659,329,285,750
1,SpanNER,Native,0.659048,0.320667,0.431421,346,179,733
2,BERT,Width ≤4,0.536705,0.313632,0.395909,329,284,720
3,SpanNER,Width ≤4,0.659048,0.329838,0.439644,346,179,703


### Final M1–M4 conformal evaluation

In [24]:
# Reconstruct the same width≤4 × six-type candidate universe
# used during development and conformal calibration.
bert_test_candidates = (
    build_bert_typed_candidates(
        bert_test_logits,
        wnut_test,
    )
)

spanner_test_candidates = (
    build_spanner_typed_candidates(
        spanner_test_inference,
        wnut_test,
    )
)

spanner_test_candidates = (
    add_spanner_boundary_uncertainty(
        spanner_test_candidates
    )
)


# Apply the frozen M1-M4 lambda specifications.
bert_test_scored = (
    add_frozen_method_scores(
        bert_test_candidates,
        "BERT",
    )
)

spanner_test_scored = (
    add_frozen_method_scores(
        spanner_test_candidates,
        "SpanNER",
    )
)


def threshold_lookup(
    model_name,
    method,
    target,
):
    """Retrieve the frozen independently calibrated q-hat."""
    match = conformal_threshold_df.loc[
        conformal_threshold_df[
            "Model"
        ].eq(model_name)
        & conformal_threshold_df[
            "Method"
        ].eq(method)
        & np.isclose(
            conformal_threshold_df[
                "Target Coverage"
            ],
            target,
        )
    ]

    if len(match) != 1:
        raise RuntimeError(
            "Expected exactly one frozen "
            "conformal threshold."
        )

    return float(
        match["q_hat"].iloc[0]
    )


test_full_gold_counts = (
    full_gold_counts_for_split(
        wnut_test
    )
)


def final_conformal_metrics(
    scored_candidates,
    model_name,
):
    """Evaluate frozen prediction sets on the official WNUT test."""
    sentence_count = len(
        wnut_test
    )

    sentence_ids = (
        scored_candidates[
            "sentence_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    is_gold = (
        scored_candidates[
            "is_gold_candidate"
        ]
        .to_numpy(
            dtype=bool
        )
    )

    # Number of width≤4 gold entities available in each sentence.
    representable_gold_count = (
        np.bincount(
            sentence_ids[
                is_gold
            ],
            minlength=sentence_count,
        )
    )

    # Full-gold coverage additionally requires that no native
    # gold entity lies outside the width≤4 candidate universe.
    structural_mask = (
        representable_gold_count
        == test_full_gold_counts
    )

    rows = []

    for method in METHODS:
        score_values = (
            scored_candidates[
                f"score_{method}"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        for target in TARGET_COVERAGES:
            q_hat = threshold_lookup(
                model_name,
                method,
                target,
            )

            selected = (
                score_values
                <= q_hat
            )

            # Number of typed candidates retained per sentence.
            set_size = np.bincount(
                sentence_ids[
                    selected
                ],
                minlength=sentence_count,
            )

            # Number of representable gold candidates retained.
            included_gold = np.bincount(
                sentence_ids[
                    selected
                    & is_gold
                ],
                minlength=sentence_count,
            )

            representable_covered = (
                included_gold
                == representable_gold_count
            )

            full_covered = (
                representable_covered
                & structural_mask
            )

            rows.append({
                "Model": model_name,
                "Method": method,
                "Target Coverage": float(
                    target
                ),
                "q_hat": float(
                    q_hat
                ),
                "Representable-Gold Coverage": float(
                    representable_covered.mean()
                ),
                "Full-Gold Coverage": float(
                    full_covered.mean()
                ),
                "Mean Set Size": float(
                    set_size.mean()
                ),
                "Median Set Size": float(
                    np.median(
                        set_size
                    )
                ),
                "Mean Non-Gold Candidates": float(
                    np.mean(
                        set_size
                        - included_gold
                    )
                ),
            })

    return pd.DataFrame(rows)


bert_final_conformal_df = (
    final_conformal_metrics(
        bert_test_scored,
        "BERT",
    )
)

spanner_final_conformal_df = (
    final_conformal_metrics(
        spanner_test_scored,
        "SpanNER",
    )
)

final_wnut_conformal_df = (
    pd.concat(
        [
            bert_final_conformal_df,
            spanner_final_conformal_df,
        ],
        ignore_index=True,
    )
)


print(
    "Final WNUT M1-M4 conformal results:"
)

display(
    final_wnut_conformal_df.round(6)
)


print(
    "\nPrimary 90% comparison:"
)

final_wnut_90_df = (
    final_wnut_conformal_df.loc[
        np.isclose(
            final_wnut_conformal_df[
                "Target Coverage"
            ],
            0.90,
        )
    ]
    .reset_index(drop=True)
)

display(
    final_wnut_90_df.round(6)
)


# Save the final results and candidate universes needed
# for the short post-hoc M4 mechanism analysis.
FINAL_TEST_OUTPUT_DIR = Path(
    "/kaggle/working/"
    "09_WNUT17_External_Replication/"
    "final_test"
)

FINAL_TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

final_wnut_baseline_df.to_csv(
    FINAL_TEST_OUTPUT_DIR
    / "wnut_final_ner_results.csv",
    index=False,
)

final_wnut_conformal_df.to_csv(
    FINAL_TEST_OUTPUT_DIR
    / "wnut_final_conformal_results.csv",
    index=False,
)

bert_test_scored.to_parquet(
    FINAL_TEST_OUTPUT_DIR
    / "bert_test_typed_candidates.parquet",
    index=False,
)

spanner_test_scored.to_parquet(
    FINAL_TEST_OUTPUT_DIR
    / "spanner_test_typed_candidates.parquet",
    index=False,
)

print(
    "\nFinal WNUT evaluation complete. "
    "No further method tuning is permitted."
)

Final WNUT M1-M4 conformal results:


,Model,Method,Target Coverage,q_hat,Representable-Gold Coverage,Full-Gold Coverage,Mean Set Size,Median Set Size,Mean Non-Gold Candidates
0,BERT,M1,0.90,0.986917,0.890443,0.870241,17.784771,6.0,17.097902
1,BERT,M1,0.95,0.995553,0.938617,0.916084,37.447552,15.0,36.701632
2,BERT,M1,0.99,0.998472,0.967366,0.944833,78.401709,47.0,77.621601
3,BERT,M2,0.90,0.990774,0.898213,0.878011,15.443667,7.0,14.749029
4,BERT,M2,0.95,0.998584,0.940171,0.917638,26.561772,16.0,25.816628
5,BERT,M2,0.99,1.010972,0.978244,0.954934,181.672883,152.0,180.881119
6,BERT,M3,0.90,0.992797,0.905983,0.885781,15.671329,8.0,14.965812
7,BERT,M3,0.95,0.998584,0.940948,0.918415,25.861694,17.0,25.115773
8,BERT,M3,0.99,1.041956,0.986014,0.962704,299.641803,240.0,298.842269
9,BERT,M4,0.90,0.990774,0.899767,0.879565,17.287490,7.0,16.591298



Primary 90% comparison:


,Model,Method,Target Coverage,q_hat,Representable-Gold Coverage,Full-Gold Coverage,Mean Set Size,Median Set Size,Mean Non-Gold Candidates
0,BERT,M1,0.9,0.986917,0.890443,0.870241,17.784771,6.0,17.097902
1,BERT,M2,0.9,0.990774,0.898213,0.878011,15.443667,7.0,14.749029
2,BERT,M3,0.9,0.992797,0.905983,0.885781,15.671329,8.0,14.965812
3,BERT,M4,0.9,0.990774,0.899767,0.879565,17.287490,7.0,16.591298
4,SpanNER,M1,0.9,0.991038,0.892774,0.872572,11.254856,6.0,10.577312
5,SpanNER,M2,0.9,0.993635,0.895882,0.875680,10.815074,6.0,10.133644
6,SpanNER,M3,0.9,0.995120,0.894328,0.874126,10.403263,7.0,9.721057
7,SpanNER,M4,0.9,0.993635,0.896659,0.876457,10.961150,7.0,10.277389



Final WNUT evaluation complete. No further method tuning is permitted.


## 11. Post-Hoc M4 Tie-Ambiguity Mechanism Check

> **Post-hoc exploratory analysis:** the WNUT-17 test set has already been evaluated. Results in this section are therefore used only to investigate M4's mechanism and are not treated as additional held-out evidence.

The original SpanNER uncertainty signal can become large when a weak candidate is strongly dominated by a neighbouring competitor. A short mechanism check therefore asks whether M4 would behave differently if uncertainty were highest for genuine probability near-ties instead.

Three controls are compared with the frozen primary results:

- **M3-selected:** the already-selected fixed M3 configuration;
- **Fixed@M4-λ:** a fixed M3-style score using M4's frozen λ values, isolating the effect of the uncertainty term;
- **Original M4:** the primary uncertainty-adaptive formulation;
- **M4-Tie:** replaces competitor-dominance uncertainty with near-tie ambiguity;
- **M4-Tie×Confidence:** additionally gates tie ambiguity by candidate confidence.

No λ values are retuned. Because the score definition changes, each exploratory variant receives a fresh conformal threshold fitted on the same 201-sentence conformal-calibration population. Evaluation then uses the already-seen WNUT test candidate universe.

In [25]:
POSTHOC_TARGETS = [0.90, 0.95]

POSTHOC_VARIANTS = [
    "M3-selected",
    "Fixed@M4-lambda",
    "Original M4",
    "M4-Tie",
    "M4-Tie×Confidence",
]


def clipped_probability_tie(probability):
    """
    Convert a probability-like ambiguity signal into a tie score.
    Maximum ambiguity occurs at 0.5.
    """
    probability = np.asarray(
        probability,
        dtype=np.float64,
    )

    return np.clip(
        1.0
        - 2.0 * np.abs(
            probability - 0.5
        ),
        0.0,
        1.0,
    )


def spanner_tie_signal(
    current_probability,
    competitor_probability,
):
    """
    Tie ambiguity is high when current and competitor probabilities
    are similar, rather than when one strongly dominates the other.
    """
    current = np.asarray(
        current_probability,
        dtype=np.float64,
    )

    competitor = np.asarray(
        competitor_probability,
        dtype=np.float64,
    )

    tie = np.zeros(
        len(current),
        dtype=np.float64,
    )

    has_competitor = np.isfinite(
        competitor
    )

    relative_difference = np.zeros(
        len(current),
        dtype=np.float64,
    )

    np.divide(
        np.abs(
            current - competitor
        ),
        (
            current
            + competitor
            + BOUNDARY_EPSILON
        ),
        out=relative_difference,
        where=has_competitor,
    )

    tie[has_competitor] = (
        1.0
        - relative_difference[
            has_competitor
        ]
    )

    return np.clip(
        tie,
        0.0,
        1.0,
    )


def add_posthoc_tie_signals(
    frame,
    model_name,
):
    """Add architecture-specific tie and confidence-gated tie signals."""
    result = frame.copy()

    if model_name == "BERT":
        # Original BERT start uncertainty is I/(B+I).
        result[
            "tie_start_uncertainty"
        ] = clipped_probability_tie(
            result[
                "start_uncertainty"
            ].to_numpy(
                dtype=np.float64
            )
        )

        # Original BERT end uncertainty is P(I-type at e+1).
        result[
            "tie_end_uncertainty"
        ] = clipped_probability_tie(
            result[
                "end_uncertainty"
            ].to_numpy(
                dtype=np.float64
            )
        )

    elif model_name == "SpanNER":
        # Use the corrected calibrated/current candidate probability.
        current_probability = result[
            "base_confidence"
        ].to_numpy(
            dtype=np.float64
        )

        result[
            "tie_start_uncertainty"
        ] = spanner_tie_signal(
            current_probability,
            result[
                "start_competitor_probability"
            ].to_numpy(
                dtype=np.float64
            ),
        )

        result[
            "tie_end_uncertainty"
        ] = spanner_tie_signal(
            current_probability,
            result[
                "end_competitor_probability"
            ].to_numpy(
                dtype=np.float64
            ),
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    confidence = result[
        "base_confidence"
    ].to_numpy(
        dtype=np.float64
    )

    result[
        "tie_conf_start_uncertainty"
    ] = (
        confidence
        * result[
            "tie_start_uncertainty"
        ].to_numpy(
            dtype=np.float64
        )
    )

    result[
        "tie_conf_end_uncertainty"
    ] = (
        confidence
        * result[
            "tie_end_uncertainty"
        ].to_numpy(
            dtype=np.float64
        )
    )

    return result


def fixed_boundary_score(
    frame,
    lambda_start,
    lambda_end,
):
    """Fixed M3-style boundary score."""
    return (
        frame[
            "base_nonconformity"
        ].to_numpy(
            dtype=np.float64
        )
        + (
            float(lambda_start)
            * frame[
                "start_rank_penalty"
            ].to_numpy(
                dtype=np.float64
            )
            + float(lambda_end)
            * frame[
                "end_rank_penalty"
            ].to_numpy(
                dtype=np.float64
            )
        )
        / 2.0
    )


def adaptive_boundary_score(
    frame,
    lambda_start,
    lambda_end,
    start_uncertainty_column,
    end_uncertainty_column,
):
    """M4-style adaptive boundary score for a chosen uncertainty signal."""
    return (
        frame[
            "base_nonconformity"
        ].to_numpy(
            dtype=np.float64
        )
        + (
            float(lambda_start)
            * (
                1.0
                - frame[
                    start_uncertainty_column
                ].to_numpy(
                    dtype=np.float64
                )
            )
            * frame[
                "start_rank_penalty"
            ].to_numpy(
                dtype=np.float64
            )
            + float(lambda_end)
            * (
                1.0
                - frame[
                    end_uncertainty_column
                ].to_numpy(
                    dtype=np.float64
                )
            )
            * frame[
                "end_rank_penalty"
            ].to_numpy(
                dtype=np.float64
            )
        )
        / 2.0
    )


def add_posthoc_scores(
    frame,
    model_name,
):
    """Construct the five post-hoc comparison scores."""
    result = add_posthoc_tie_signals(
        frame,
        model_name,
    )

    m3 = method_parameters[
        model_name
    ]["M3"]

    m4 = method_parameters[
        model_name
    ]["M4"]

    result[
        "posthoc_M3-selected"
    ] = fixed_boundary_score(
        result,
        m3["lambda_start"],
        m3["lambda_end"],
    )

    result[
        "posthoc_Fixed@M4-lambda"
    ] = fixed_boundary_score(
        result,
        m4["lambda_start"],
        m4["lambda_end"],
    )

    result[
        "posthoc_Original M4"
    ] = adaptive_boundary_score(
        result,
        m4["lambda_start"],
        m4["lambda_end"],
        "start_uncertainty",
        "end_uncertainty",
    )

    result[
        "posthoc_M4-Tie"
    ] = adaptive_boundary_score(
        result,
        m4["lambda_start"],
        m4["lambda_end"],
        "tie_start_uncertainty",
        "tie_end_uncertainty",
    )

    result[
        "posthoc_M4-Tie×Confidence"
    ] = adaptive_boundary_score(
        result,
        m4["lambda_start"],
        m4["lambda_end"],
        "tie_conf_start_uncertainty",
        "tie_conf_end_uncertainty",
    )

    return result


bert_posthoc_calibration = (
    add_posthoc_scores(
        bert_conformal_scored,
        "BERT",
    )
)

spanner_posthoc_calibration = (
    add_posthoc_scores(
        spanner_conformal_scored,
        "SpanNER",
    )
)

bert_posthoc_test = (
    add_posthoc_scores(
        bert_test_scored,
        "BERT",
    )
)

spanner_posthoc_test = (
    add_posthoc_scores(
        spanner_test_scored,
        "SpanNER",
    )
)

print(
    "Post-hoc tie-ambiguity scores constructed."
)

Post-hoc tie-ambiguity scores constructed.


### Recalibrate and evaluate the post-hoc variants

In [26]:
def calibrate_posthoc_variant(
    frame,
    score_column,
    target,
):
    """Fit a fresh sentence-level q-hat for one exploratory score."""
    sentence_count = len(
        wnut_conformal_calibration
    )

    sentence_scores = (
        frame.loc[
            frame[
                "is_gold_candidate"
            ],
            [
                "sentence_id",
                score_column,
            ],
        ]
        .groupby(
            "sentence_id"
        )[score_column]
        .max()
        .reindex(
            range(sentence_count)
        )
        .fillna(0.0)
        .to_numpy(
            dtype=np.float64
        )
    )

    return finite_sample_q(
        sentence_scores,
        target,
    )


posthoc_qhat_rows = []

for model_name, calibration_frame in [
    (
        "BERT",
        bert_posthoc_calibration,
    ),
    (
        "SpanNER",
        spanner_posthoc_calibration,
    ),
]:
    for variant in POSTHOC_VARIANTS:
        score_column = (
            f"posthoc_{variant}"
        )

        for target in POSTHOC_TARGETS:
            posthoc_qhat_rows.append({
                "Model": model_name,
                "Variant": variant,
                "Target Coverage": target,
                "q_hat": calibrate_posthoc_variant(
                    calibration_frame,
                    score_column,
                    target,
                ),
            })


posthoc_qhat_df = pd.DataFrame(
    posthoc_qhat_rows
)


def evaluate_posthoc_variant(
    frame,
    model_name,
    variant,
    target,
):
    """Evaluate one post-hoc variant on the already-seen test universe."""
    threshold_row = posthoc_qhat_df.loc[
        posthoc_qhat_df[
            "Model"
        ].eq(model_name)
        & posthoc_qhat_df[
            "Variant"
        ].eq(variant)
        & np.isclose(
            posthoc_qhat_df[
                "Target Coverage"
            ],
            target,
        )
    ]

    q_hat = float(
        threshold_row[
            "q_hat"
        ].iloc[0]
    )

    score = frame[
        f"posthoc_{variant}"
    ].to_numpy(
        dtype=np.float64
    )

    sentence_ids = frame[
        "sentence_id"
    ].to_numpy(
        dtype=np.int64
    )

    is_gold = frame[
        "is_gold_candidate"
    ].to_numpy(
        dtype=bool
    )

    sentence_count = len(
        wnut_test
    )

    representable_gold_count = (
        np.bincount(
            sentence_ids[
                is_gold
            ],
            minlength=sentence_count,
        )
    )

    selected = (
        score <= q_hat
    )

    set_size = np.bincount(
        sentence_ids[
            selected
        ],
        minlength=sentence_count,
    )

    included_gold = np.bincount(
        sentence_ids[
            selected & is_gold
        ],
        minlength=sentence_count,
    )

    covered = (
        included_gold
        == representable_gold_count
    )

    return {
        "Model": model_name,
        "Variant": variant,
        "Target Coverage": float(
            target
        ),
        "q_hat": q_hat,
        "Representable-Gold Coverage": float(
            covered.mean()
        ),
        "Mean Set Size": float(
            set_size.mean()
        ),
        "Median Set Size": float(
            np.median(
                set_size
            )
        ),
    }


posthoc_result_rows = []

for model_name, test_frame in [
    (
        "BERT",
        bert_posthoc_test,
    ),
    (
        "SpanNER",
        spanner_posthoc_test,
    ),
]:
    for variant in POSTHOC_VARIANTS:
        for target in POSTHOC_TARGETS:
            posthoc_result_rows.append(
                evaluate_posthoc_variant(
                    test_frame,
                    model_name,
                    variant,
                    target,
                )
            )


posthoc_results_df = pd.DataFrame(
    posthoc_result_rows
)

print(
    "Post-hoc 90% mechanism comparison:"
)

posthoc_90_df = (
    posthoc_results_df.loc[
        np.isclose(
            posthoc_results_df[
                "Target Coverage"
            ],
            0.90,
        )
    ]
    .reset_index(drop=True)
)

display(
    posthoc_90_df.round(6)
)

Post-hoc 90% mechanism comparison:


,Model,Variant,Target Coverage,q_hat,Representable-Gold Coverage,Mean Set Size,Median Set Size
0,BERT,M3-selected,0.9,0.992797,0.905983,15.671329,8.0
1,BERT,Fixed@M4-lambda,0.9,0.990814,0.898213,15.506605,7.0
2,BERT,Original M4,0.9,0.990774,0.899767,17.287490,7.0
3,BERT,M4-Tie,0.9,0.988150,0.888112,16.125097,7.0
4,BERT,M4-Tie×Confidence,0.9,0.990814,0.898990,15.556333,7.0
5,SpanNER,M3-selected,0.9,0.995120,0.894328,10.403263,7.0
6,SpanNER,Fixed@M4-lambda,0.9,0.996817,0.890443,10.102564,7.0
7,SpanNER,Original M4,0.9,0.993635,0.896659,10.961150,7.0
8,SpanNER,M4-Tie,0.9,0.995120,0.898213,10.835276,6.0
9,SpanNER,M4-Tie×Confidence,0.9,0.996817,0.891997,10.135975,7.0


## 12. Conclusion

This notebook replicated the complete boundary-uncertainty and conformal-prediction pipeline on WNUT-17 using its native six-type entity ontology.

On the official test set, SpanNER achieved stronger strict typed-span NER performance than BERT:

- **BERT:** F1 = 0.3887 natively and 0.3959 under width≤4 support;
- **SpanNER:** F1 = 0.4314 natively and 0.4396 under width≤4 support.

At the nominal 90% conformal target, the uncertainty-adaptive M4 score did not establish a coverage-efficiency advantage over the fixed alternatives.

For BERT, M3 achieved the highest representable-gold coverage among M1–M4 at **0.9060**, with a mean prediction-set size of **15.67**. M4 achieved lower coverage of **0.8998** while producing a larger mean set of **17.29**.

For SpanNER, M4 achieved representable-gold coverage of **0.8967** with a mean set size of **10.96**. M2 achieved very similar coverage (**0.8959**) with a smaller set (**10.82**), while M3 produced the smallest fixed-method set (**10.40**) at coverage **0.8943**. M4 therefore did not dominate the fixed alternatives on both coverage and efficiency.

The post-hoc mechanism analysis provided additional evidence that the semantics of the uncertainty signal matter. For SpanNER, replacing competitor-dominance uncertainty with a near-tie formulation slightly improved both coverage and set size relative to Original M4. However, the exploratory tie variants still did not clearly dominate the selected fixed M3 configuration. Because this analysis was performed after the WNUT test set had already been observed, it is treated only as a mechanism diagnostic and possible direction for future work.

Overall, the external replication supports the dissertation's central finding: boundary uncertainty remains informative across architectures and domains, but directly using it to relax boundary penalties in conformal scoring does not reliably improve the coverage-efficiency trade-off over simpler fixed boundary-aware methods.